# Demanda eléctrica — VS Code (ejecución local)

Versión 1.3. Lectura local por hojas, seis modelos y escenarios.

**Uso:** abra la carpeta raíz `Prodma` en VS Code, seleccione el kernel `.venv` y ejecute las celdas en orden. Los cuatro históricos se leen desde `datahistorica/`, las altas desde `nuevosclientes/` y cada ejecución escribe una carpeta independiente dentro de `resultados/`. Los archivos de entrada nunca se modifican.

Las fechas originales se conservan como fin de intervalo. No se afirma que `kW tot` sea una media: la energía derivada requiere confirmación. La proyección larga es un escenario de perfil y crecimiento, no una predicción ML validada a diez años. Las nuevas cargas pendientes no se suman silenciosamente.

## Actualización 1.3 — Migración de Colab a VS Code

Se eliminaron el montaje de Google Drive, la autenticación interactiva y la descarga mediante Drive API. La raíz del proyecto se descubre desde el directorio de trabajo (raíz o carpeta `notebook`) o mediante la variable opcional `PRODMA_ROOT`. El inventario valida exactamente cuatro archivos históricos locales y un archivo de nuevos clientes antes de iniciar el procesamiento.

## Actualización 1.1 — Nuevos clientes constantes

Cada etapa anual se suma al código exacto de **Medidor**, no al nombre de alimentador. Desde su fecha de ingreso se mantiene constante las 24 horas, en cada intervalo de 15 minutos, y se acumula con las etapas anteriores. No se aplican crecimiento, rampa ni factores reductores. Una etapa de 200 kW añade **200 kW** a cada punto del perfil base, no 50 kW; su energía por intervalo sería 50 kWh. Q se calcula con el factor de potencia del archivo.

La matriz operativa exportada contiene el **total**; se conserva también la matriz base y la contribución individual por medidor/variable. Las tasas de los escenarios afectan solo la base. Las fechas y la confirmación de que una etapa no está ya en el histórico siguen pendientes en AJUSTES_CLIENTES cuando no se conocen; no se supone automáticamente un día del año. Los códigos sin coincidencia se reportan y nunca se reasignan por similitud.


## 01 · Dependencias

Antes de abrir el notebook, cree/active `.venv` e instale una vez `requirements.txt`. La celda siguiente solo valida el entorno; no modifica paquetes durante una ejecución reproducible.

In [ ]:
import sys, subprocess, importlib.util

MODULOS_REQUERIDOS = {
    'numpy':'numpy', 'pandas':'pandas', 'scipy':'scipy', 'xlrd':'xlrd',
    'openpyxl':'openpyxl', 'pyarrow':'pyarrow', 'xlsxwriter':'xlsxwriter',
    'sklearn':'scikit-learn', 'statsmodels':'statsmodels', 'xgboost':'xgboost',
    'lightgbm':'lightgbm', 'catboost':'catboost', 'holidays':'holidays',
    'matplotlib':'matplotlib', 'seaborn':'seaborn', 'psutil':'psutil',
    'joblib':'joblib', 'prophet':'prophet', 'torch':'torch'
}
faltantes = [paquete for modulo,paquete in MODULOS_REQUERIDOS.items()
             if importlib.util.find_spec(modulo) is None]
if faltantes:
    raise RuntimeError(
        'Faltan dependencias en el kernel actual: '+', '.join(faltantes)+
        '. En la terminal de VS Code ejecute: .\.venv\Scripts\python.exe -m pip install -r requirements.txt'
    )
print('Entorno local validado:', sys.executable)

## 02 · Configuración

Entradas editables. No se inventan fechas ni tasas. Escenarios vacíos producen solo la base, hasta completar hipótesis.


In [ ]:
from pathlib import Path
import os, re, json, gc, time, hashlib, unicodedata, warnings, traceback
import numpy as np
import pandas as pd
import xlrd, openpyxl, joblib, psutil
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
CFG = dict(modo='rapido', seed=42,
           max_medidores_rapido=2, train_days_rapido=120, train_days_completo=None,
           train_frac=0.8, es_days=14, test_days=28, cv_days=7, cv_folds=3, min_train_days=35,
           fecha_inicio='2023-01-01 00:15', frecuencia='15min', tolerancia_seg=10,
           fin_plan='2028-01-01 00:00',  # cierre de 2027 (fin de intervalo; 2037-01-01 00:00 era el cierre de 2036)
           confirmar_potencias_medias=False, incorporar_clientes=True,
           calendario_etapas='incremental', mes_ingreso=None, dia_ingreso=None,
           escenarios={}, exportar_excel=True,
           alfa_significancia=0.05, rango_durbin_watson=(1.5,2.5), r2_minimo_aceptable=0.90,
           usar_temperatura_climatologica=True, n_trials_optuna=12,
           ga_poblacion=16, ga_generaciones=10, usar_pbi=True, usar_poblacion=True,
           # --- Cierre 2027 / reanudación sin reentrenar (ver docs/MANUAL_EJECUCION.md) ---
           reanudar_run=None,        # ruta de una corrida previa: continúa ahí, sin entrenar
           max_dias_sin_datos=180,   # series sin dato en más de N días respecto al último global: no se proyectan
           umbral_sanidad=0.25,      # desvío máx. de la media proyectada vs. últimos 12 meses antes de marcar REVISAR
           workers_pronostico=6,     # procesos para el pronóstico recursivo (cada uno carga su modelo: cuidar la RAM)
           backtest_horizontes=(96,672,2688),  # el mayor decide el método de cada serie
           margen_ml=0.95,           # el ML solo se usa si su RMSE a 28 días es < margen x el del mejor perfil
           max_medidores_adicionales=None)
# train_days_completo=None: en modo 'completo' se entrena con TODO el historico
# disponible de cada medidor (no una ventana arbitraria mas corta que el dataset
# cargado, ej. ~3 anios y 7 meses); train_days_rapido=120 si mantiene una ventana
# corta para pruebas rapidas.
# train_frac=0.8: del historico disponible de cada medidor (ya acotado por
# train_days_rapido/completo), el 80% cronologico mas antiguo se usa para
# entrenar y el 20% final para evaluar honestamente (celda 14); ya no se usa
# una cantidad fija de dias (test_days) para ese corte.
# El pronostico operativo (celda 20) ya no tiene un horizonte corto separado
# (forecast_days_rapido/completo): siempre proyecta, con el modelo ML ganador,
# cada 15 min desde el ultimo dato observado hasta CFG['fin_plan'], guardando
# un archivo por año calendario (resto de este año, luego año por año).
# es_days=14: dentro del 80% de entrenamiento (train_frac), las ultimas 2 semanas se
# reservan EXCLUSIVAMENTE para el early stopping de XGBoost/LightGBM/CatBoost (celda 14,
# 20 y 21) -- nunca se usan para ajustar arboles/pesos ni para elegir hiperparametros;
# solo para decidir cuando dejar de agregar rondas de boosting.
# Ejemplo SOLO para editar y sustentar: CFG['escenarios']={'bajo':0.01,'base':0.03,'alto':0.05}
# Confirmar calendario_etapas='incremental' o 'acumulado' y fechas para activar nuevos clientes.
# Ajustes por clave Cliente|Año: {'fecha':'2027-07-01','kw':..., 'incluido_hist':False,
# 'confirmado':True, 'medidor':'AL209'}
# Nuevos clientes: incremento constante cada 15 min; perfil/rampa/tasas no se aplican.
# alfa_significancia: nivel de significancia para Ljung-Box, Breusch-Pagan, White, F y
# t-Student (celdas 15 y 16). rango_durbin_watson: sin autocorrelacion de primer orden
# si el estadistico DW cae dentro de este rango (regla practica estandar: 2=ausencia de
# autocorrelacion; <1.5 indica autocorrelacion positiva, >2.5 negativa). r2_minimo_aceptable:
# umbral preferente de R2 fuera de muestra (celda 15, los 6 modelos + benchmarks) y de R2
# ajustado en el OLS de referencia (celda 16). Ningun umbral descarta un modelo ni detiene
# la ejecucion de otros medidores: el incumplimiento se reporta en las tablas CSV
# correspondientes y se registra como ADVERTENCIA en el log (nunca se oculta).
# usar_temperatura_climatologica: agrega en features() (celda 12) una columna de
# temperatura ESPERADA (mediana historica por mes/hora, variables_externas/temperatura/,
# reanalisis Open-Meteo 2023-2026 para 'ica_centro') como exogena. Es un supuesto
# climatologico declarado -- NO un pronostico meteorologico real -- y se usa IGUAL en
# entrenamiento y en el pronostico recursivo hasta CFG['fin_plan'] (2037), donde no
# existe temperatura futura real; requiere haber corrido antes
# tools/generar_variables_externas.py. Poner en False para desactivarlo.
# ga_poblacion/ga_generaciones: algoritmo genetico que elige, por serie (medidor,
# variable), el subconjunto de columnas de features() que mejor generaliza en CV
# temporal (evaluado con un Ridge simple, barato de entrenar) -- reutilizado por los
# 6 modelos ML de esa serie; los benchmarks Estacional_* no lo usan.
# n_trials_optuna: numero de combinaciones de hiperparametros que Optuna prueba por
# modelo (busqueda bayesiana TPE, reemplaza la grilla exhaustiva fija de antes) --
# mismo presupuesto aproximado que antes (grillas de 4-6 combinaciones) pero explora
# un espacio CONTINUO en vez de solo los puntos fijos de una grilla pequeña.
AJUSTES_CLIENTES = {}
PERFILES = {'continuo':np.ones(96),
            'comercial':np.array([1.0 if 8<=h/4<20 else 0.15 for h in range(96)]),
            'industrial':np.array([1.0 if 6<=h/4<22 else 0.3 for h in range(96)])}
# Sin perfil confirmado no se asigna uno automáticamente.
np.random.seed(CFG['seed'])
assert CFG['modo'] in ('rapido','completo')


## 03 · Rutas locales y resultados

Descubre la raíz del proyecto de forma portable, valida su estructura y crea una carpeta de resultados independiente. Puede definir `PRODMA_ROOT` si el kernel se inicia fuera del proyecto.

In [ ]:
def encontrar_raiz_proyecto(inicio=None):
    override=os.environ.get('PRODMA_ROOT')
    if override:
        candidatos=[Path(override).expanduser().resolve()]
    else:
        actual=Path(inicio or Path.cwd()).resolve()
        candidatos=[actual,*actual.parents]
    for candidato in candidatos:
        if (candidato/'datahistorica').is_dir() and (candidato/'nuevosclientes').is_dir():
            return candidato
    raise FileNotFoundError(
        'No se encontró la raíz de Prodma. Abra la carpeta del proyecto en VS Code '
        'o defina la variable de entorno PRODMA_ROOT.'
    )

RAIZ_PROYECTO=encontrar_raiz_proyecto()
DIRECTORIO_HISTORICO=RAIZ_PROYECTO/'datahistorica'
DIRECTORIO_CLIENTES=RAIZ_PROYECTO/'nuevosclientes'
ROOT=RAIZ_PROYECTO/'resultados'; ROOT.mkdir(parents=True,exist_ok=True)
CACHE=ROOT/'cache_v1'; CACHE.mkdir(parents=True,exist_ok=True)
# Nombre descriptivo: <modo>-<backend>_YYYYMMDD_HHMMSS (ej: R-AUT_20260917_114523)
modo_tag = CFG.get('modo', 'rapido')[0].upper()  # R=Rapido, C=Completo
backend_tag = CFG.get('backend', 'auto')[:3].upper()  # AUT/CPU/GPU
# Reanudación: con CFG['reanudar_run'] se CONTINÚA en la carpeta de una corrida previa (sus
# modelos/tablas/figuras se reutilizan, nada se reentrena) en vez de crear una nueva.
REANUDANDO=bool(CFG.get('reanudar_run'))
if REANUDANDO:
    RUN=Path(CFG['reanudar_run'])
    if not RUN.is_absolute(): RUN=RAIZ_PROYECTO/RUN
    assert (RUN/'modelos').is_dir() and (RUN/'tablas/metricas.csv').exists(), f'{RUN} no parece una corrida previa'
else:
    RUN=ROOT/(f'{modo_tag}-{backend_tag}_{pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")}')
for p in ('tablas','figuras','modelos','pronosticos','logs'):
    (RUN/p).mkdir(parents=True,exist_ok=True)
if REANUDANDO:
    (RUN/f'config_reanudacion_{pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")}.json').write_text(
        json.dumps(CFG,indent=2,ensure_ascii=False,default=str),encoding='utf-8')
else:
    (RUN/'config.json').write_text(json.dumps(CFG,indent=2,ensure_ascii=False),encoding='utf-8')
    (RUN/'entorno.txt').write_text(
        subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True),encoding='utf-8')
LOG=[]
if REANUDANDO and (RUN/'logs/eventos.csv').exists():
    LOG=pd.read_csv(RUN/'logs/eventos.csv',dtype=str,keep_default_na=False).to_dict('records')  # conserva los eventos previos
    LOG=[e for k,e in enumerate(LOG) if e not in LOG[:k]]   # sin duplicados de reanudaciones anteriores
def log(stage,item,status,detail=''):
    evento=dict(etapa=stage,item=str(item),estado=status,detalle=str(detail))
    if REANUDANDO and evento in LOG: return   # una reanudación no repite eventos ya registrados
    LOG.append(evento)
    pd.DataFrame(LOG).to_csv(RUN/'logs/eventos.csv',index=False)
print('Proyecto:',RAIZ_PROYECTO)
print('Resultados:',RUN)
if REANUDANDO: print('REANUDACIÓN sobre una corrida previa: no se reentrena ningún modelo ML.')


## 04 · Inventario local

Exige cuatro históricos (`demanda2023` a `demanda2026`) y un archivo de nuevos clientes. Acepta `.xls` o `.xlsx`, detecta duplicados y nunca elige un archivo ambiguo silenciosamente.

In [ ]:
# Cuatro históricos anuales (nombre base exacto, extensión .xls o .xlsx)
# y un archivo de nuevos clientes detectado por patrón dentro de su carpeta dedicada.
HIST_BASES=[f'demanda{year}' for year in range(2023,2027)]
PATRON_CLIENTES=re.compile(r'^(?=.*nuevos)(?=.*client).*\.xlsx?$|^(?=.*demanda)(?=.*solicitad).*\.xlsx?$',re.IGNORECASE)

archivos_historicos=[p for p in DIRECTORIO_HISTORICO.rglob('*')
                     if p.is_file() and p.suffix.lower() in ('.xls','.xlsx')]
archivos_clientes=[p for p in DIRECTORIO_CLIENTES.rglob('*')
                   if p.is_file() and p.suffix.lower() in ('.xls','.xlsx')]
todos_los_archivos=archivos_historicos+archivos_clientes
files=[dict(name=p.name,path=str(p.resolve()),size=p.stat().st_size,
            modifiedTime=pd.Timestamp(p.stat().st_mtime,unit='s').isoformat())
       for p in todos_los_archivos]
inventory=pd.DataFrame(files); display(inventory)
inventory.to_csv(RUN/'tablas/inventario_archivos.csv',index=False)

assert PATRON_CLIENTES.match('nuevosclientes.xlsx') and PATRON_CLIENTES.match('Nuevos_Clientes_2026.xlsx')
assert PATRON_CLIENTES.match('Demanda_Solicitada.xlsx') and not PATRON_CLIENTES.match('demanda2025.xls')

selected={}; ESTADO_ARCHIVOS=[]
for base in HIST_BASES:
    patron_hist=re.compile(re.escape(base)+r'\.xlsx?$',re.IGNORECASE)
    matches=[f for f in files if Path(f['path']).parent==DIRECTORIO_HISTORICO.resolve()
             and patron_hist.fullmatch(f['name'])]
    if len(matches)==1:
        nombre_normalizado=base+Path(matches[0]['name']).suffix.lower()
        selected[nombre_normalizado]=matches[0]; estado='encontrado'
    elif len(matches)==0:
        estado='faltante'; nombre_normalizado=base+'.xls/.xlsx'
    else:
        raise ValueError(f'{base}: se esperaba 1 archivo, encontrados {len(matches)}: {[m["path"] for m in matches]}')
    ESTADO_ARCHIVOS.append(dict(archivo=nombre_normalizado,estado=estado,tipo='historico_anual'))

client_matches=[f for f in files if Path(f['path']).is_relative_to(DIRECTORIO_CLIENTES.resolve())
                and PATRON_CLIENTES.fullmatch(f['name'])]
if len(client_matches)==1:
    CLIENTE_ARCHIVO_KEY='nuevosclientes'+Path(client_matches[0]['name']).suffix.lower()
    selected[CLIENTE_ARCHIVO_KEY]=client_matches[0]; estado_clientes='encontrado'
    nombre_clientes=client_matches[0]['name']
elif len(client_matches)==0:
    estado_clientes='faltante'; nombre_clientes='(patrón *nuevos*clientes*.xls* o *demanda*solicitada*.xls*)'
    CLIENTE_ARCHIVO_KEY=None
else:
    raise ValueError('Archivo de nuevos clientes ambiguo: '+str([f['path'] for f in client_matches]))
ESTADO_ARCHIVOS.append(dict(archivo=nombre_clientes,estado=estado_clientes,tipo='nuevos_clientes'))

estado_archivos=pd.DataFrame(ESTADO_ARCHIVOS); display(estado_archivos)
estado_archivos.to_csv(RUN/'tablas/estado_archivos_requeridos.csv',index=False)
FALTANTES=[r['archivo'] for r in ESTADO_ARCHIVOS if r['estado']=='faltante']
HISTORICOS_FALTANTES=[r['archivo'] for r in ESTADO_ARCHIVOS
                      if r['estado']=='faltante' and r['tipo']=='historico_anual']
CLIENTES_DISPONIBLE=(estado_clientes=='encontrado')
for f in FALTANTES:
    log('inventario',f,'FALTANTE','No encontrado en las carpetas locales configuradas')
if not selected:
    raise ValueError('Ningún archivo requerido está disponible; no es posible el diagnóstico')
if FALTANTES:
    print('ADVERTENCIA — archivos locales ausentes (diagnóstico parcial):',*FALTANTES,sep='\n - ')
else:
    print('Los cinco archivos requeridos están presentes.')
print('Disponibles para procesar:',list(selected.keys()))

## 05 · Validación de archivos locales

Resuelve las rutas absolutas, confirma que cada entrada sea un archivo regular no vacío y conserva el tamaño para auditoría. No se copia ni modifica ningún Excel de entrada.

In [ ]:
paths={}
for name,meta in selected.items():
    path=Path(meta['path']).expanduser().resolve()
    if not path.is_file():
        raise FileNotFoundError(f'Archivo local no encontrado: {path}')
    if path.stat().st_size<=0:
        raise ValueError(f'Archivo local vacío: {path}')
    if path.stat().st_size!=int(meta['size']):
        raise ValueError(f'El archivo cambió durante el inventario: {path}')
    paths[name]=path
    print(name,'->',path,round(path.stat().st_size/1e6,1),'MB verificados')

## 06 · Funciones temporales y esquema

Local Time tiene prioridad. Se preserva fin de intervalo; el desfase UTC se audita.


In [ ]:
def norm(s):
    return re.sub(r'[^a-z0-9]+',' ',unicodedata.normalize('NFKD',str(s)).encode('ascii','ignore').decode().lower()).strip()
def key(s): return re.sub(r'[^A-Za-z0-9]+','_',str(s)).strip('_')
ALIASES={'P':{'kw tot','kw tot mean'},'Q':{'kvar tot','kvar tot mean'},
         'S_medida':{'kva tot','kva tot mean','kva'},'Fecha':{'local time'},'UTC':{'timestamp'}}
from prodma.excel_schema import find_first_columns
def find_cols(headers):
    # Política global: ante encabezados duplicados, conservar siempre la primera
    # columna de izquierda a derecha para todos los archivos y todas las hojas.
    return find_first_columns(headers,ALIASES,required={'Fecha','P','Q'})
def abrir_libro(path):
    # Históricos y archivo de clientes pueden llegar en .xls o .xlsx (el proveedor los
    # cambia sin previo aviso); xlrd>=2 solo lee .xls, y openpyxl solo .xlsx -- se elige
    # el motor por extensión real del archivo, nunca se asume uno fijo.
    ext=Path(path).suffix.lower()
    if ext=='.xls': return 'xls',xlrd.open_workbook(str(path),on_demand=True)
    if ext=='.xlsx': return 'xlsx',openpyxl.load_workbook(str(path),read_only=True,data_only=True)
    raise ValueError(f'Extensión no soportada: {ext} (solo .xls o .xlsx)')
def nombres_hojas(tipo,wb):
    return list(wb.sheet_names()) if tipo=='xls' else list(wb.sheetnames)
def filas_hoja(tipo,wb,sn):
    # Generador de filas (lista de valores por fila), mismo contrato para ambos motores.
    if tipo=='xls':
        sh=wb.sheet_by_name(sn)
        for r in range(sh.nrows): yield sh.row_values(r)
    else:
        for fila in wb[sn].iter_rows(values_only=True): yield list(fila)
def datemode_libro(tipo,wb):
    # xlsx: sistema de fechas 1900 estándar, equivalente a datemode=0 de xlrd (sin la
    # ambigüedad 1900/1904 que sí existe en .xls).
    return wb.datemode if tipo=='xls' else 0
def descargar_hoja(tipo,wb,sn):
    if tipo=='xls': wb.unload_sheet(sn)
    # openpyxl en modo read_only ya libera la hoja al terminar de recorrerla.
def cerrar_libro(tipo,wb):
    if tipo=='xls': wb.release_resources()
    else: wb.close()
def excel_dates(values,datemode):
    def one(v):
        if v is None or v=='': return pd.NaT
        try:
            if isinstance(v,(float,int,np.number)): return xlrd.xldate_as_datetime(float(v),datemode)
            return pd.to_datetime(v,dayfirst=True,errors='coerce')
        except (ValueError,OverflowError): return pd.NaT
    return pd.DatetimeIndex([one(v) for v in values])
def align_dates(idx,tolerance=10):
    rounded=idx.round('15min'); delta=np.abs((idx-rounded).total_seconds())
    valid=idx.notna() & (delta<=tolerance)
    return rounded,valid,delta
def metrics(y,p):
    y=np.asarray(y,float);p=np.asarray(p,float);m=np.isfinite(y)&np.isfinite(p);y=y[m];p=p[m]
    if not len(y): return {'n':0,'RMSE':np.nan}
    e=y-p; ss=np.sum((y-y.mean())**2);nz=np.abs(y)>1e-6;den=np.abs(y)+np.abs(p)
    return dict(n=len(y),RMSE=float(np.sqrt(np.mean(e*e))),MAE=float(np.mean(abs(e))),
        MAPE=float(np.mean(abs(e[nz]/y[nz]))*100) if nz.any() else np.nan,
        MAPE_excluidos=int((~nz).sum()),sMAPE=float(np.mean(np.divide(200*abs(e),den,out=np.zeros_like(e),where=den>0))),
        WAPE=float(100*sum(abs(e))/sum(abs(y))) if sum(abs(y)) else np.nan,
        R2=float(1-sum(e*e)/ss) if ss else np.nan,sesgo=float(e.mean()))

## 07 · Pruebas de funciones

Estas pruebas se ejecutan antes de leer masivamente los libros.


In [ ]:
assert find_cols(['Local Time','kVAR tot','kW tot'])['P']==2
assert find_cols(['Local Time','kW tot mean','kVAR tot mean'])['Q']==2
assert excel_dates([44927.0104166667],0)[0].round('s')==pd.Timestamp('2023-01-01 00:15')
_,ok,_=align_dates(pd.DatetimeIndex(['2023-01-01 17:15:05','2023-01-01 17:18']))
assert ok.tolist()==[True,False]
assert metrics([0,0],[0,0])['MAPE_excluidos']==2
assert metrics([1,2,3],[1,2,3])['RMSE']==0
assert abs(np.sqrt(3**2+4**2)-5)<1e-12
print('7 comprobaciones iniciales superadas')

## 08 · Lectura y caché hoja por hoja

Inspecciona todas las hojas. No se utiliza sheet_name=None. Conflictos duplicados quedan nulos y auditados.


In [ ]:
INDEX=[]; SCHEMAS=[]; ISSUES=[]
for name,path in paths.items():
    if not re.fullmatch(r'demanda\d{4}\.xlsx?',name): continue
    meta=selected[name]; fingerprint=meta.get('md5Checksum') or str(meta['modifiedTime'])
    tipo,wb=abrir_libro(path)
    try:
        for sn in nombres_hojas(tipo,wb):
            if sn.startswith('|PRIVATE|'):
                SCHEMAS.append(dict(archivo=name,hoja=sn,estado='interna')); continue
            ck=hashlib.sha256(f"{fingerprint}|{sn}|parser_v2|{CFG['tolerancia_seg']}".encode()).hexdigest()[:24]
            dest=CACHE/(ck+'.parquet'); audit=CACHE/(ck+'.json')
            try:
                if dest.exists() and audit.exists():
                    info=json.loads(audit.read_text())
                else:
                    hs=None; header=None; filas_datos=[]
                    for i,fila in enumerate(filas_hoja(tipo,wb,sn)):
                        if header is None:
                            if i>=20: break
                            if 'local time' in [norm(x) for x in fila]: header=i; hs=fila
                            continue
                        filas_datos.append(fila)
                    if header is None: raise ValueError('Encabezado no encontrado')
                    cols=find_cols(hs)
                    raw={k:[(fila[c] if c<len(fila) else None) for fila in filas_datos] for k,c in cols.items()}
                    datemode=datemode_libro(tipo,wb)
                    original=excel_dates(raw.pop('Fecha'),datemode)
                    dt,valid,delta=align_dates(original,CFG['tolerancia_seg'])
                    utc=excel_dates(raw.pop('UTC'),datemode) if 'UTC' in raw else None
                    df=pd.DataFrame({k:pd.to_numeric(pd.Series(v),errors='coerce').to_numpy() for k,v in raw.items()},index=dt)
                    df.index.name='Fecha'; df=df.loc[valid].sort_index()
                    duplicates=df.index.duplicated(keep=False)
                    if duplicates.any():
                        df.loc[duplicates].to_csv(RUN/'tablas'/f'duplicados_{ck}.csv')
                        # Valor único por fecha/variable; contradicciones no se promedian.
                        df=df.groupby(level=0).agg(lambda v: v.dropna().iloc[0] if v.dropna().nunique()==1 else np.nan)
                    info=dict(archivo=name,hoja=sn,estado='ok',filas=len(df),columnas=hs,
                         inicio=str(df.index.min()),fin=str(df.index.max()),P_validos=int(df.P.notna().sum()),
                         Q_validos=int(df.Q.notna().sum()),desalineados=int((~valid).sum()),
                         redondeados=int(((delta>0)&valid).sum()),duplicados=int(duplicates.sum()),
                         UTC_desfase_h=float(np.nanmedian((utc-original).total_seconds()/3600)) if utc is not None else None,
                         P_media=norm(hs[cols['P']]).endswith('mean'),Q_media=norm(hs[cols['Q']]).endswith('mean'))
                    if len(df):
                        info['intervalos_ausentes']=len(pd.date_range(df.index.min(),df.index.max(),freq='15min'))-len(df)
                    df.to_parquet(dest);audit.write_text(json.dumps(info,ensure_ascii=False))
                SCHEMAS.append(info); INDEX.append(dict(archivo=name,medidor=sn,path=str(dest),info=info))
            except Exception as exc:
                log('lectura',f'{name}/{sn}','ERROR',exc); ISSUES.append((name,sn,str(exc)))
            finally:
                descargar_hoja(tipo,wb,sn);gc.collect()
    finally: cerrar_libro(tipo,wb)
    print(name,'procesado; RAM MB',round(psutil.Process().memory_info().rss/1e6))
pd.DataFrame(SCHEMAS).to_csv(RUN/'tablas/inventario_hojas.csv',index=False)
assert INDEX,'Ninguna hoja válida'

## 09 · Matriz histórica

Una hoja Excel, fecha única, P/Q por medidor. No se agregan barras y alimentadores como si fueran cargas independientes. Los medidores con CERO mediciones válidas de P y Q en todo el histórico disponible (hojas presentes pero nunca instrumentadas) se excluyen aquí -- antes de construir la matriz y las figuras -- y quedan registrados como ADVERTENCIA en el log.

Además se generan las gráficas de integración total (consolidado): energía integrada por medidor (barras, todo el histórico) y demanda P consolidada (suma de todos los medidores en el tiempo). También se grafica la curva P/Q observada de cada medidor individual (una figura por medidor).


In [ ]:
# Medidores SIN ninguna medicion real (P_validos=0 en TODOS los anios, ej. hojas
# presentes en el Excel pero nunca instrumentadas): se excluyen aqui, antes de la
# matriz y las figuras -- no tiene sentido construir columnas 100% NaN ni graficar
# una curva vacia para un medidor del que no existe NINGUN dato observado. Distinto
# del umbral de elegibilidad para ENTRENAR (celda 14): aqui es 'no existe el dato',
# alla es 'existe pero es insuficiente para modelar con confianza'.
_VALIDOS_POR_MEDIDOR={}
for x in INDEX:
    _VALIDOS_POR_MEDIDOR[x['medidor']]=_VALIDOS_POR_MEDIDOR.get(x['medidor'],0)+x['info']['P_validos']+x['info']['Q_validos']
_SIN_MEDICIONES=sorted(m for m,v in _VALIDOS_POR_MEDIDOR.items() if v==0)
for m in _SIN_MEDICIONES:
    log('medidor_sin_datos',m,'ADVERTENCIA','Excluido: 0 mediciones validas de P y Q en todo el historico disponible')
if _SIN_MEDICIONES:
    print('Medidores excluidos por no tener NINGUNA medicion real:',', '.join(_SIN_MEDICIONES))
MEDIDORES=sorted({x['medidor'] for x in INDEX if x['medidor'] not in _SIN_MEDICIONES}); TOKENS={m:key(m) for m in MEDIDORES}
assert len(set(TOKENS.values()))==len(TOKENS),'Colisión de nombres: corregir catálogo'
# Caché en memoria: load_meter se invoca sin caché en 9 puntos distintos del notebook
# (celdas 09,10,11,14,16,20,21,22,25); sin memoria, cada llamada relee los parquet,
# concatena, revisa duplicados y reindexa al rango completo desde cero para el mismo
# medidor una y otra vez. Con caché cada medidor se construye una sola vez; el resultado
# es idéntico (misma función pura sobre INDEX/RUN fijos), solo evita el trabajo repetido.
_METER_CACHE={}
def load_meter(m):
    if m in _METER_CACHE: return _METER_CACHE[m]
    parts=[pd.read_parquet(x['path']) for x in INDEX if x['medidor']==m]
    df=pd.concat(parts).sort_index()
    if df.index.duplicated().any():
        df[df.index.duplicated(False)].to_csv(RUN/'tablas'/f'conflictos_anuales_{TOKENS[m]}.csv')
        df=df.groupby(level=0).agg(lambda v:v.dropna().iloc[0] if v.dropna().nunique()==1 else np.nan)
    out=df.reindex(pd.date_range(df.index.min(),df.index.max(),freq='15min',name='Fecha'))
    _METER_CACHE[m]=out
    return out
end=max(pd.Timestamp(x['info']['fin']) for x in INDEX)
grid=pd.date_range(CFG['fecha_inicio'],end,freq='15min',name='Fecha')
# Cache de la matriz integrada: evita reconstruir columna por columna (concat de 50
# medidores) en reinicios que no cambian los datos fuente ni esta logica. Clave = hash
# de los ck (ya estables, seccion 09) de TODOS los parquet anuales + version.
CACHE_MATRIZ=RAIZ_PROYECTO/'resultados'/'cache_matriz_v1'; CACHE_MATRIZ.mkdir(parents=True,exist_ok=True)
_ck_matriz=hashlib.sha256(('|'.join(sorted(Path(x['path']).stem for x in INDEX))
    +f"|matriz_v1|{CFG['fecha_inicio']}|{CFG['confirmar_potencias_medias']}").encode()).hexdigest()[:24]
_dest_matriz=CACHE_MATRIZ/(_ck_matriz+'.parquet')
if _dest_matriz.exists():
    matrix=pd.read_parquet(_dest_matriz)
    print('Matriz consolidada recuperada del cache (misma fuente de datos, sin reconstruir)')
else:
    # Construcción por diccionario + concat único (evita fragmentar el DataFrame insertando
    # columna por columna en el bucle; resultado numéricamente idéntico, sin PerformanceWarning).
    columnas={}
    for _i,m in enumerate(MEDIDORES,1):
        print(f'Matriz: {m} ({_i}/{len(MEDIDORES)})',flush=True)
        d=load_meter(m)
        for var in ('P','Q'): columnas[f'{TOKENS[m]}_{var}']=d[var].reindex(grid).astype('float32')
        if 'S_medida' in d.columns:
            columnas[f'{TOKENS[m]}_S_medida']=d['S_medida'].reindex(grid).astype('float32')
        flags=[x['info'] for x in INDEX if x['medidor']==m]
        for var in ('P','Q'):
            if CFG['confirmar_potencias_medias'] or all(x[var+'_media'] for x in flags):
                columnas[f'{TOKENS[m]}_E{var}_derivada']=columnas[f'{TOKENS[m]}_{var}']*.25
    matrix=pd.concat(columnas,axis=1)
    matrix.index.name='Fecha'
    matrix.to_parquet(_dest_matriz)
# Curvas por medidor (P y Q observados, serie completa): una figura por medidor, ademas
# de la consolidada (mas abajo). Permite auditar visualmente cada medidor individual.
for _i,m in enumerate(MEDIDORES,1):
    if REANUDANDO and (RUN/f'figuras/curva_{TOKENS[m]}.png').exists(): continue  # (reanudación) figura ya generada
    print(f'Figura: {m} ({_i}/{len(MEDIDORES)})',flush=True)
    d=load_meter(m)
    # Panel de S medida (KVA) solo si el medidor la tiene (no todas las hojas traen
    # esa columna); no se agrega un panel vacio para los que no la tienen.
    tiene_s='S_medida' in d.columns and d['S_medida'].notna().any()
    # Panel unico (no subplots apilados): P, Q y S_medida superpuestos con distinto
    # color en el mismo eje, para comparar las tres curvas de un vistazo (ej. si un
    # salto en P coincide con un salto en Q/S, o si S se aparta de P+Q).
    fig,ax=plt.subplots(figsize=(14,5))
    d['P'].plot(ax=ax,lw=.5,color='tab:blue',label='P (kW)')
    d['Q'].plot(ax=ax,lw=.5,color='tab:green',label='Q (kvar)')
    if tiene_s:
        d['S_medida'].plot(ax=ax,lw=.5,color='tab:purple',label='S medido (kVA)')
    ax.set_ylabel('Potencia (kW / kvar / kVA)');ax.set_title(f'{m} — P, Q y S observados')
    ax.legend(loc='upper right')
    fig.tight_layout();fig.savefig(RUN/f'figuras/curva_{TOKENS[m]}.png',dpi=120);plt.close(fig)
matrix.to_parquet(RUN/'tablas/Matriz_Consolidada.parquet')
if CFG['exportar_excel'] and not (REANUDANDO and (RUN/'tablas/Matriz_Consolidada.xlsx').exists()):
    assert len(matrix)<1048576 and len(matrix.columns)<16384
    matrix.to_excel(RUN/'tablas/Matriz_Consolidada.xlsx',sheet_name='Matriz_Consolidada',engine='xlsxwriter')
pd.DataFrame({'Medidor':MEDIDORES,'Codigo':[TOKENS[m] for m in MEDIDORES]}).to_csv(RUN/'tablas/catalogo_medidores.csv',index=False)
# --- Integración total (consolidado): energía por medidor y demanda total sumada ---
p_cols=[c for c in matrix.columns if c.endswith('_P')]
if p_cols:
    energia_medidor=(matrix[p_cols]*.25).sum().rename(index=lambda c:c[:-2]).sort_values(ascending=False)
    energia_medidor.to_csv(RUN/'tablas/integracion_energia_por_medidor.csv',header=['energia_kwh'])
    fig,ax=plt.subplots(figsize=(max(8,.4*len(energia_medidor)),6))
    energia_medidor.plot(kind='bar',ax=ax,color='steelblue')
    ax.set_ylabel('Energía integrada (kWh)');ax.set_title('Integración total de energía por medidor (histórico completo)')
    ax.tick_params(axis='x',rotation=90);fig.tight_layout()
    fig.savefig(RUN/'figuras/integracion_energia_por_medidor.png',dpi=140);plt.close(fig)
    total_P=matrix[p_cols].sum(axis=1,min_count=1)
    total_P.rename('P_total_kw').to_csv(RUN/'tablas/demanda_consolidada_total.csv')
    fig,ax=plt.subplots(figsize=(14,5))
    total_P.plot(ax=ax,lw=.6,color='darkorange')
    ax.set_ylabel('Demanda consolidada P (kW)');ax.set_title('Demanda consolidada de todos los medidores (suma, serie completa)')
    fig.tight_layout();fig.savefig(RUN/'figuras/demanda_consolidada_total.png',dpi=140);plt.close(fig)
display(matrix.head());del matrix;gc.collect()

## 10 · Calidad e imputación causal

Solo relleno corto con pasado; los objetivos imputados no se puntúan. Se generan boxplots por medidor (P y Q) para visualizar la dispersión y los atípicos observados (referencia visual, sección independiente del recorte). Cuando el medidor trae S_medida (KVA) real (45/50, verificado), se limpia con el MISMO criterio (relleno causal + recorte local por MAD) que P y Q -- así puede entrar al entrenamiento (sección 14) con la misma calidad que P/Q. El recorte usa un límite LOCAL (mediana ± 6·MAD móvil causal, ventana de 7 días), no un límite fijo sobre toda la historia: un límite global aplanaría tramos enteros con demanda naturalmente más alta (crecimiento real, temporada alta). Además, solo se recorta un punto si es ATÍPICO PUNTUAL (aislado) — si la mayoría de sus vecinos cercanos (±2h) también se apartan del nivel local, se trata como un cambio de nivel real y no se toca. Nunca se eliminan valores, solo se reducen al límite más cercano; el valor bruto (sin recortar) se conserva en '<var>_bruto' para auditoría, y el detalle de límites y puntos recortados queda en 'recorte_atipicos_boxplot.csv'. Además se grafica, por medidor y variable, la curva del dataset limpio (bruto vs. con recorte aplicado).


In [ ]:
def causal_fill(s):
    s=s.replace([np.inf,-np.inf],np.nan).astype(float)
    out=s.ffill(limit=4)
    # Semana anterior observada: sin recurrencia que fabrique meses enteros.
    out=out.fillna(s.shift(672))
    flags=pd.Series(np.where(s.notna(),0,np.where(out.notna(),3,5)),index=s.index)
    if s.first_valid_index() is not None: out.loc[out.index<s.first_valid_index()]=np.nan
    return out,flags
# Bloxplots por variable (observados, todos los medidores): visualizan la dispersión
# y los valores atípicos antes de cualquier corrección.
for var in ('P','Q'):
    datos=[load_meter(m)[var].dropna().values for m in MEDIDORES]
    fig,ax=plt.subplots(figsize=(max(8,.4*len(MEDIDORES)),6))
    ax.boxplot(datos,showfliers=True)
    ax.set_xticklabels([TOKENS[m] for m in MEDIDORES])
    ax.set_ylabel(f'{var} observado');ax.set_title(f'Bloxplot de {var} por medidor (valores observados)')
    ax.tick_params(axis='x',rotation=90);fig.tight_layout()
    fig.savefig(RUN/f'figuras/boxplot_{var}_por_medidor.png',dpi=140);plt.close(fig)

# Cache del dataset limpio por medidor: evita rehacer el computo pesado (rolling
# median/MAD sobre ~129k filas x hasta 3 variables) en reinicios que no cambian los
# datos fuente ni esta logica de limpieza. Clave = hash de los ck (seccion 09, ya
# estables) de los parquet anuales de ESE medidor + version. Las figuras SIEMPRE se
# regeneran (baratas, y consistente con reemplazar figuras cada corrida): se derivan
# de las columnas _bruto/limpio que ya estan en el DataFrame, sea cache hit o miss.
CACHE_LIMPIO=RAIZ_PROYECTO/'resultados'/'cache_limpio_v1'; CACHE_LIMPIO.mkdir(parents=True,exist_ok=True)
def _fingerprint_limpieza(m):
    cks=sorted(Path(x['path']).stem for x in INDEX if x['medidor']==m)
    return hashlib.sha256(('|'.join(cks)+'|limpieza_v1').encode()).hexdigest()[:24]

QUALITY=[];RECORTES=[];COMPARACION_S=[]
for m in MEDIDORES:
    ck_limpio=_fingerprint_limpieza(m)
    dest_limpio=CACHE_LIMPIO/(ck_limpio+'.parquet');audit_limpio=CACHE_LIMPIO/(ck_limpio+'.json')
    if dest_limpio.exists() and audit_limpio.exists():
        out=pd.read_parquet(dest_limpio);stats=json.loads(audit_limpio.read_text())
    else:
        df=load_meter(m); out=df.copy()
        # S_medida (KVA) se limpia IGUAL que P/Q cuando el medidor la trae real -- para
        # que pueda entrenarse despues (celda 14) con la misma calidad, no datos crudos.
        tiene_s_real=('S_medida' in df.columns) and df['S_medida'].notna().any()
        variables_medidor=('P','Q')+(('S_medida',) if tiene_s_real else ())
        _recortes_m=[];_quality_m=[]
        for var in variables_medidor:
            s=df[var];filled,calidad=causal_fill(s)
            median=s.rolling(672,min_periods=96).median().shift(1)
            mad=(s-median).abs().rolling(672,min_periods=96).median().shift(1)
            mad_valido=mad.replace(0,np.nan)
            lo=median-6*1.4826*mad_valido;hi=median+6*1.4826*mad_valido
            candidato=(s-median).abs()>6*1.4826*mad_valido
            # Puntual vs. sostenido: si LA MAYORIA de los vecinos cercanos (+/-2h, 17
            # pasos) TAMBIEN se apartan del nivel local, es un cambio de nivel real
            # (crecimiento, entrada de un cliente, temporada alta) -- NO un dato erroneo
            # puntual, y no se recorta. Solo se recorta el punto AISLADO cuyos vecinos
            # siguen en el nivel normal. Limitacion conocida: justo despues (hasta ~7
            # dias) de un cambio de nivel real sostenido, el MAD movil queda inflado por
            # la propia transicion y puede tardar en volver a detectar picos puntuales
            # con la sensibilidad habitual.
            proporcion_vecinos=candidato.rolling(17,center=True,min_periods=1).mean()
            atipico=(candidato&(proporcion_vecinos<0.5)).fillna(False)
            recortado=atipico.reindex(filled.index,fill_value=False)
            out[var+'_bruto']=filled
            valor_recortado=filled.clip(lower=lo,upper=hi)
            out[var]=filled.where(~recortado,valor_recortado)
            out[var+'_calidad']=calidad
            out[var+'_atipico']=atipico
            out[var+'_recortado']=recortado
            n_rec=int(recortado.sum())
            _recortes_m.append(dict(medidor=m,variable=var,
                limite_inferior_mediana=float(lo.median()) if lo.notna().any() else np.nan,
                limite_superior_mediana=float(hi.median()) if hi.notna().any() else np.nan,
                puntos_recortados=n_rec))
            _quality_m.append(dict(medidor=m,variable=var,observados=int(s.notna().sum()),
                 nulos=int(s.isna().sum()),negativos=int((s<0).sum()),ceros=int((s==0).sum()),
                 media=s.mean(),std=s.std(),P95=s.quantile(.95),maximo=s.max(),
                 fecha_max=str(s.idxmax()) if s.notna().any() else None,
                 factor_carga=s.mean()/s.max() if s.max()>0 else np.nan,
                 recortados_boxplot=n_rec))
        out['S_calculada']=np.hypot(out.P,out.Q)
        out['FP_calculado']=out.P/out.S_calculada.replace(0,np.nan)
        # S medida (KVA tot/KVA tot mean/KVA, si la hoja la trae) vs S calculada=hypot(P,Q):
        # una discrepancia grande es evidencia real de un problema de medicion o de
        # convencion de signo en P/Q (no solo ruido de redondeo); se calcula aqui y se
        # registra como ADVERTENCIA SIEMPRE (cache hit o miss, ver mas abajo).
        if 'S_medida' in df.columns:
            diferencia_pct=((df['S_medida']-out['S_calculada']).abs()/out['S_calculada'].replace(0,np.nan))*100
            dif_mediana=float(diferencia_pct.median()) if diferencia_pct.notna().any() else np.nan
            _comparacion_s_m=dict(medidor=m,S_medida_disponible=True,
                diferencia_pct_mediana=dif_mediana,S_medida_observados=int(df['S_medida'].notna().sum()))
        else:
            _comparacion_s_m=dict(medidor=m,S_medida_disponible=False,
                diferencia_pct_mediana=np.nan,S_medida_observados=0)
        out.to_parquet(dest_limpio)
        stats=dict(quality=_quality_m,recortes=_recortes_m,comparacion_s=_comparacion_s_m)
        audit_limpio.write_text(json.dumps(stats,ensure_ascii=False,default=float))
    QUALITY.extend(stats['quality']);RECORTES.extend(stats['recortes']);COMPARACION_S.append(stats['comparacion_s'])
    dif_mediana=stats['comparacion_s'].get('diferencia_pct_mediana')
    if dif_mediana is not None and pd.notna(dif_mediana) and dif_mediana>10:
        log('S_medida_vs_calculada',m,'ADVERTENCIA',
            f'Diferencia mediana {dif_mediana:.1f}% entre KVA medido y sqrt(P^2+Q^2) calculado')
    # Curva del dataset limpio (bruto vs limpio): SIEMPRE fresca, derivada de las
    # columnas _bruto/limpio ya presentes en out (cache hit o miss da lo mismo aqui).
    variables_out=[c[:-6] for c in out.columns if c.endswith('_bruto')]
    for var in variables_out:
        if REANUDANDO and (RUN/f'figuras/limpio_{TOKENS[m]}_{var}.png').exists(): continue  # (reanudación) figura ya generada
        n_rec=int(out[var+'_recortado'].sum()) if var+'_recortado' in out.columns else 0
        fig,ax=plt.subplots(figsize=(14,4))
        out[var+'_bruto'].plot(ax=ax,lw=.4,color='lightgray',label='bruto (relleno causal, sin recortar)')
        out[var].plot(ax=ax,lw=.6,color='tab:blue',label='limpio (solo atipicos puntuales recortados)')
        ax.legend(loc='upper right');ax.set_ylabel(var);ax.set_title(f'{m} — {var}: dataset limpio vs bruto ({n_rec} puntos recortados)')
        fig.tight_layout();fig.savefig(RUN/f'figuras/limpio_{TOKENS[m]}_{var}.png',dpi=120);plt.close(fig)
    out.to_parquet(RUN/'tablas'/f'limpio_{TOKENS[m]}.parquet')
pd.DataFrame(QUALITY).to_csv(RUN/'tablas/calidad_estadistica.csv',index=False)
pd.DataFrame(RECORTES).to_csv(RUN/'tablas/recorte_atipicos_boxplot.csv',index=False)
pd.DataFrame(COMPARACION_S).to_csv(RUN/'tablas/comparacion_S_medida_calculada.csv',index=False)

## 11 · Correlaciones

Valores observados, no series artificialmente rellenadas. Pearson y Spearman separadas para P/Q.


In [ ]:
for var in ('P','Q'):
    mat=pd.concat({m:load_meter(m)[var] for m in MEDIDORES},axis=1)
    for method in ('pearson','spearman'):
        corr=mat.corr(method=method,min_periods=96)
        corr.to_csv(RUN/'tablas'/f'correlacion_{var}_{method}.csv')
        fig,ax=plt.subplots(figsize=(14,12));sns.heatmap(corr,vmin=-1,vmax=1,cmap='coolwarm',ax=ax)
        ax.set_title(f'{var} — {method}, observados');fig.tight_layout()
        fig.savefig(RUN/'figuras'/f'correlacion_{var}_{method}.png',dpi=140);plt.close(fig)
    del mat;gc.collect()

## 12 · Variables para pronóstico

Objetivo en t; predictores calculados solo con t−1 y anteriores.

**Temperatura climatológica (`CFG['usar_temperatura_climatologica']`):** se agrega una columna de temperatura *esperada* (mediana histórica por mes/hora, calculada sobre `variables_externas/temperatura/`) como exógena, aplicada de forma **idéntica** en entrenamiento y en el pronóstico recursivo hasta `CFG['fin_plan']`. No es un pronóstico meteorológico real ni una medición: es un supuesto climatológico declarado, necesario porque no existe temperatura futura observada para un horizonte de años. Ver `variables_externas/README.md` para la cobertura y limitaciones de la fuente (Open-Meteo, reanálisis ERA5, una sola coordenada representativa de Ica).

**Estacionalidad no sinusoidal:** la hora del día usa 3 armónicos de Fourier (k=1,2,3), no solo el fundamental -- una sola sinusoide no representa la rampa/meseta/pico real de la demanda y deja autocorrelación residual detectable en Ljung-Box (lag=96/672) aunque Durbin-Watson (lag=1) no la vea. El día de la semana se codifica como 6 dummies (lunes implícito) en vez de un seno/coseno suave, para permitir curvas de carga distintas por tipo de día.

**Señal semanal directa:** `media_mismo_dia_hora_4sem`/`std_mismo_dia_hora_4sem` promedian el mismo día y hora exactos de las últimas 4 semanas (t-7d,-14d,-21d,-28d) -- más directo que un rolling(672) genérico que mezcla los 7 días sin distinguir cuál.

**Confiabilidad del historial reciente:** `prop_imputado_96` es la proporción de valores imputados (no observados) en las últimas 24h, para que el modelo pueda distinguir lags/rolling calculados sobre dato real de los calculados sobre relleno causal.


In [ ]:
import holidays
FERIADOS=holidays.country_holidays('PE',years=range(2023,2037))

# Temperatura climatologica (mes,hora)->mediana, sobre observaciones reales de
# variables_externas/temperatura/ (Open-Meteo, reanalisis ERA5, 2023-2026, 'ica_centro').
# Se usa como EXOGENA en TODO el horizonte -- pasado y futuro hasta CFG['fin_plan'] -- de
# forma CONSISTENTE en entrenamiento y en el pronostico recursivo: es un supuesto
# climatologico declarado, nunca un pronostico meteorologico real (ver celda 12 y
# variables_externas/README.md). Aplicarla solo en el futuro y no en entrenamiento
# introduciria una discrepancia entre la señal que ve el modelo al entrenar y la que
# recibe al pronosticar.
import sys
_SRC_LOCAL=RAIZ_PROYECTO/'src'
if str(_SRC_LOCAL) not in sys.path: sys.path.insert(0,str(_SRC_LOCAL))
from prodma.variables_externas import (construir_climatologia_mensual_horaria, temperatura_climatologica,
    cargar_pbi_mensual, pbi_en_fecha, generar_poblacion_ica_interpolada, poblacion_en_fecha)
CLIMATOLOGIA_TEMPERATURA=None
if CFG.get('usar_temperatura_climatologica'):
    _archivos_temp_horaria=sorted((RAIZ_PROYECTO/'variables_externas'/'temperatura').glob('temperatura_horaria_*.csv'))
    if not _archivos_temp_horaria:
        raise RuntimeError(
            "CFG['usar_temperatura_climatologica']=True pero no existe ningun archivo en "
            "variables_externas/temperatura/. Ejecute antes: "
            ".\\.venv\\Scripts\\python.exe tools\\generar_variables_externas.py "
            "o ponga CFG['usar_temperatura_climatologica']=False."
        )
    _temp_obs=pd.concat([pd.read_csv(p) for p in _archivos_temp_horaria],ignore_index=True)
    CLIMATOLOGIA_TEMPERATURA=construir_climatologia_mensual_horaria(_temp_obs)
    print('Climatologia de temperatura construida desde',len(_archivos_temp_horaria),
          'archivo(s) horario(s); rango mediana:',
          round(CLIMATOLOGIA_TEMPERATURA.min(),1),'a',round(CLIMATOLOGIA_TEMPERATURA.max(),1),'°C')

# PBI mensual (BCRPData, serie confirmada por el usuario) como exogena: proxy de
# actividad economica NACIONAL (no regional de Ica), mismo criterio de consistencia
# entrenamiento/pronostico que la temperatura. Mas alla del ultimo dato real (BCRP
# publica con rezago), se mantiene el ultimo valor conocido -- ver pbi_en_fecha().
PBI_MENSUAL=None
if CFG.get('usar_pbi'):
    _archivos_pbi=sorted((RAIZ_PROYECTO/'variables_externas'/'economia').glob('pbi_mensual_bcrp_*.csv'))
    if not _archivos_pbi:
        raise RuntimeError(
            "CFG['usar_pbi']=True pero no existe ningun archivo en variables_externas/economia/. "
            "Ejecute antes: .\\.venv\\Scripts\\python.exe tools\\generar_variables_externas.py "
            "o ponga CFG['usar_pbi']=False."
        )
    PBI_MENSUAL=cargar_pbi_mensual(_archivos_pbi[-1])
    print('PBI mensual cargado:',len(PBI_MENSUAL),'meses,',PBI_MENSUAL.index.min().date(),
          'a',PBI_MENSUAL.index.max().date())

# Poblacion de Ica (INEI, 2 puntos reales + interpolacion/extrapolacion declarada,
# ver variables_externas/poblacion/README.md) como exogena, calculada directamente
# hasta el fin del horizonte de planificacion (no requiere archivo: es barato de
# calcular y asi cubre CFG['fin_plan'] sin regenerar nada).
POBLACION_ANUAL=None
if CFG.get('usar_poblacion'):
    _anio_fin_poblacion=pd.Timestamp(CFG['fin_plan']).year
    _df_poblacion=generar_poblacion_ica_interpolada(2022,_anio_fin_poblacion)
    POBLACION_ANUAL=pd.Series(_df_poblacion['poblacion'].to_numpy(),
        index=_df_poblacion['fecha'].str[:4].astype(int))
    print('Poblacion de Ica calculada 2022-'+str(_anio_fin_poblacion)+
          ' (observados INEI:',int((_df_poblacion['dato']=='observado_inei').sum()),'de',len(_df_poblacion),')')

def features(s):
    z,flags=causal_fill(s);idx=z.index;X=pd.DataFrame(index=idx)
    # 2do y 3er armonico del ciclo horario: la demanda real tiene rampas/mesetas, no
    # es una sinusoide pura -- con un solo armonico (k=1) el modelo repite el mismo
    # error sistematico a la misma hora todos los dias (Ljung-Box en lag=96/672 lo
    # detecta aunque Durbin-Watson en lag=1 pase); k=2,3 capturan esa forma.
    hora_val=idx.hour+idx.minute/60
    for k in (1,2,3):
        X[f'hora_sin{k}']=np.sin(2*np.pi*k*hora_val/24);X[f'hora_cos{k}']=np.cos(2*np.pi*k*hora_val/24)
    X['anio_sin']=np.sin(2*np.pi*idx.dayofyear/365.25);X['anio_cos']=np.cos(2*np.pi*idx.dayofyear/365.25)
    # Dia de semana como categoria (no solo seno/coseno suave): el lunes no se parece
    # al sabado por una sinusoide, son curvas de carga distintas por tipo de dia.
    # Lunes (dia_0) queda como referencia implicita (evita colinealidad exacta).
    for dia in range(1,7):
        X[f'dia_{dia}']=(idx.dayofweek==dia).astype(int)
    X['feriado']=[int(t.date() in FERIADOS) for t in idx]
    X['fin_semana']=(idx.dayofweek>=5).astype(int)
    if CLIMATOLOGIA_TEMPERATURA is not None:
        X['temp_climatologica_c']=temperatura_climatologica(idx,CLIMATOLOGIA_TEMPERATURA)
    if PBI_MENSUAL is not None:
        X['pbi_bcrp']=pbi_en_fecha(idx,PBI_MENSUAL)
    if POBLACION_ANUAL is not None:
        X['poblacion_ica']=poblacion_en_fecha(idx,POBLACION_ANUAL)
    for lag in (1,4,96,672): X[f'lag_{lag}']=z.shift(lag)
    for win in (4,24,96,672):
        r=z.shift(1).rolling(win,min_periods=win)
        X[f'media_{win}']=r.mean();X[f'std_{win}']=r.std()
        X[f'mediana_{win}']=r.median();X[f'min_{win}']=r.min();X[f'max_{win}']=r.max();X[f'p95_{win}']=r.quantile(.95)
    # Promedio/std del MISMO dia y hora en las ultimas 4 semanas (t-7d,-14d,-21d,-28d):
    # senal semanal DIRECTA ("que es tipico para este dia/hora exacto"), en vez de
    # inferirla indirectamente de un rolling(672) que mezcla los 7 dias sin distinguir cual.
    mismos_dia_hora=pd.concat([z.shift(672*k) for k in (1,2,3,4)],axis=1)
    X['media_mismo_dia_hora_4sem']=mismos_dia_hora.mean(axis=1)
    X['std_mismo_dia_hora_4sem']=mismos_dia_hora.std(axis=1)
    # Proporcion de valores imputados (no observados) en las ultimas 24h: si el
    # historial reciente es mayormente relleno causal, el modelo puede aprender a
    # confiar menos en esos lags/rolling en vez de tratarlos como dato real.
    imputado=(flags>0).astype(float)
    X['prop_imputado_96']=imputado.shift(1).rolling(96,min_periods=1).mean()
    return X
def temporal_cv(index,n_splits=3,days=7):
    # Intervalos de calendario: filtrar objetivos nulos no comprime el tiempo.
    splits=[];end=index.max()
    for k in range(n_splits,0,-1):
        upper=end-pd.Timedelta(days=(k-1)*days);lower=upper-pd.Timedelta(days=days)
        tr=np.flatnonzero(index<=lower);va=np.flatnonzero((index>lower)&(index<=upper))
        if len(tr)<96*14 or len(va)<96:raise ValueError('Ventana temporal sin suficientes observados')
        splits.append((tr,va))
    return splits


## 13 · Seis modelos e hiperparámetros

Búsqueda bayesiana (Optuna, muestreador TPE) sobre un espacio CONTINUO de hiperparámetros por modelo (`CFG['n_trials_optuna']` combinaciones probadas, reemplaza la grilla exhaustiva fija de antes), evaluada en CV temporal. XGBoost/LightGBM/CatBoost tienen tope de 400 rondas de boosting pero se frenan antes con *early stopping* real (paciencia `PACIENCIA_EARLY_STOPPING=20`) sobre un tramo reservado (`CFG['es_days']`) que nunca se usa para ajustar árboles ni para elegir hiperparámetros — el early stopping también aplica dentro de cada intento (*trial*) de Optuna. El MLP usa su propio *early stopping* interno (`validation_fraction`). CPU en `N_HILOS_CPU=24` (de 32 procesadores logicos disponibles en esta maquina) para RandomForest/XGBoost/LightGBM/CatBoost -- medido con benchmark real: 3.5x mas rapido en RandomForest frente a los 2 hilos anteriores, mismo modelo exacto.


In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.compose import TransformedTargetRegressor
import optuna
from optuna.distributions import FloatDistribution, IntDistribution
from optuna_integration import OptunaSearchCV
optuna.logging.set_verbosity(optuna.logging.WARNING)  # silenciar el log de cada trial
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import lightgbm as lgb
from catboost import CatBoostRegressor
# Tope de rondas/epocas AMPLIO (400): quien decide cuanto entrena de verdad cada modelo
# iterativo es el early stopping (evaluado en el tramo reservado CFG['es_days'], celda 14),
# no un numero fijo elegido a mano. Los grids tambien se ampliaron: 2-3 valores por
# hiperparametro (antes un unico par), y se agrega learning_rate a los tres de boosting.
PACIENCIA_EARLY_STOPPING=20
# N_HILOS_CPU: medido con benchmark real sobre un dataset del tamano usado en
# produccion (100k filas, 40 features) en esta maquina (i9-14900HX, 32 procesadores
# logicos, confirmado con Get-CimInstance Win32_Processor): RandomForest n_jobs=2 ->
# 274s, n_jobs=24 -> 78s (3.5x); LightGBM n_jobs=2 -> 1.86s, n_jobs=24 -> 0.78s (~2.4x).
# Mismo modelo/arboles/random_state exactos, solo repartido entre mas nucleos -- no
# cambia ningun resultado ni reduce nada del entrenamiento. Se deja un margen (24 de
# 32) para el propio kernel de Jupyter, el sistema operativo y otras apps abiertas.
N_HILOS_CPU=24

def probar_backends_gpu():
    solicitado=CFG.get('backend','auto')
    if solicitado not in ('auto','cpu','gpu'):
        raise ValueError("CFG['backend'] debe ser auto, cpu o gpu")
    estado={'XGBoost':False,'CatBoost':False,'LightGBM':False};detalle={}
    if solicitado=='cpu':
        return estado,{k:'CPU solicitado' for k in estado}
    Xp=np.arange(128,dtype=float).reshape(64,2);yp=np.sin(np.arange(64)/10)
    pruebas={
      'XGBoost':XGBRegressor(n_estimators=4,max_depth=2,tree_method='hist',device='cuda:0',verbosity=0),
      'CatBoost':CatBoostRegressor(iterations=4,depth=2,task_type='GPU',devices='0',verbose=False),
      'LightGBM':LGBMRegressor(n_estimators=4,max_depth=2,device_type='gpu',gpu_device_id=0,verbosity=-1)}
    for nombre,modelo in pruebas.items():
        try:
            modelo.fit(Xp,yp);pred=modelo.predict(Xp[:2])
            if not np.isfinite(pred).all():raise ValueError('predicción no finita')
            estado[nombre]=True;detalle[nombre]='GPU verificada con fit/predict real'
        except Exception as exc:
            detalle[nombre]=f'{type(exc).__name__}: {exc}'
    if solicitado=='gpu' and not (estado['XGBoost'] and estado['CatBoost']):
        raise RuntimeError('GPU obligatoria no disponible para XGBoost/CatBoost: '+str(detalle))
    return estado,detalle

GPU_OK,GPU_DETALLE=probar_backends_gpu()
pd.DataFrame([{'modelo':k,'gpu_disponible':GPU_OK[k],'detalle':GPU_DETALLE[k]} for k in GPU_OK]).to_csv(
    RUN/'logs/backends_gpu.csv',index=False)
print('Backends GPU efectivos:',GPU_OK)
from prodma.transformaciones import _transformar_y,_destransformar_y  # en el paquete: los .joblib no dependen de __main__
def models():
    seed=CFG["seed"]
    # Distribuciones continuas para Optuna (busqueda bayesiana TPE), no listas fijas: el
    # rango cubre y amplia los valores de la grilla original (comentario historico:
    # Ridge alpha .1-100, RandomForest n_estimators 80-200/min_samples_leaf 2-20,
    # XGBoost/LightGBM/CatBoost profundidad chica-mediana y learning_rate .05-.1).
    def _tt(est): return TransformedTargetRegressor(regressor=est,func=_transformar_y,inverse_func=_destransformar_y)
    return {
     'Ridge':(_tt(make_pipeline(StandardScaler(),Ridge())),{'regressor__ridge__alpha':FloatDistribution(.01,200,log=True)}),
     'RandomForest':(_tt(RandomForestRegressor(max_depth=14,n_jobs=N_HILOS_CPU,random_state=seed)),
                      {'regressor__n_estimators':IntDistribution(50,300),'regressor__min_samples_leaf':IntDistribution(1,30)}),
     'XGBoost':(_tt(XGBRegressor(n_estimators=400,n_jobs=N_HILOS_CPU,random_state=seed,tree_method='hist',device=('cuda:0' if GPU_OK['XGBoost'] else 'cpu'),
                    early_stopping_rounds=PACIENCIA_EARLY_STOPPING)),
                {'regressor__max_depth':IntDistribution(2,10),'regressor__learning_rate':FloatDistribution(.01,.3,log=True)}),
     'LightGBM':(_tt(LGBMRegressor(n_estimators=400,n_jobs=N_HILOS_CPU,random_state=seed,verbosity=-1,device_type=('gpu' if GPU_OK['LightGBM'] else 'cpu'))),
                {'regressor__num_leaves':IntDistribution(7,127),'regressor__learning_rate':FloatDistribution(.01,.3,log=True)}),
     'CatBoost':(_tt(CatBoostRegressor(iterations=400,thread_count=N_HILOS_CPU,random_seed=seed,verbose=False,task_type=('GPU' if GPU_OK['CatBoost'] else 'CPU'),devices=('0' if GPU_OK['CatBoost'] else None))),
                {'regressor__depth':IntDistribution(3,10),'regressor__learning_rate':FloatDistribution(.01,.3,log=True)}),
     'MLP':(_tt(make_pipeline(StandardScaler(),MLPRegressor(hidden_layer_sizes=(32,16),max_iter=400,
          early_stopping=True,validation_fraction=.15,n_iter_no_change=15,
          shuffle=False,random_state=seed))),{'regressor__mlpregressor__alpha':FloatDistribution(1e-5,1,log=True)})}
def fit_kwargs(name,Xval,yval):
    # Early stopping real con un tramo reservado (nunca visto en el fit, celda 14/20/21):
    # frena antes del tope de rondas en cuanto el error de validacion deja de mejorar
    # durante PACIENCIA_EARLY_STOPPING rondas seguidas. RandomForest/Ridge no son iterativos
    # (nada que frenar); MLP usa su propio early stopping interno (validation_fraction arriba).
    # yval se transforma IGUAL que el target de entrenamiento: TransformedTargetRegressor
    # transforma yt internamente en .fit(), pero NO toca los fit_params (eval_set) -- sin
    # esto el early stopping compararia predicciones en escala transformada contra un
    # objetivo en escala real, una comparacion sin sentido.
    yval=_transformar_y(yval)
    if name=='XGBoost': return dict(eval_set=[(Xval,yval)],verbose=False)
    if name=='LightGBM': return dict(eval_X=Xval,eval_y=yval,callbacks=[lgb.early_stopping(PACIENCIA_EARLY_STOPPING,verbose=False)])
    if name=='CatBoost': return dict(eval_set=(Xval,yval),early_stopping_rounds=PACIENCIA_EARLY_STOPPING,verbose=False)
    return {}
def es_split(Xd,yd,nombre_modelo):
    # Reserva las ultimas CFG['es_days'] del bloque recibido EXCLUSIVAMENTE para el early
    # stopping de los tres modelos de boosting; para los demas modelos no se recorta nada
    # (no la necesitan, y asi no se les resta historial de entrenamiento sin motivo).
    if nombre_modelo not in ('XGBoost','LightGBM','CatBoost'):
        return Xd,yd,Xd.iloc[:0],yd.iloc[:0]
    n_es=min(CFG['es_days']*96,len(Xd)//4)
    if n_es<1: return Xd,yd,Xd.iloc[:0],yd.iloc[:0]
    return Xd.iloc[:-n_es],yd.iloc[:-n_es],Xd.iloc[-n_es:],yd.iloc[-n_es:]

## 14 · Selección temporal y prueba final

Se entrena P y Q para todos los medidores elegibles, y ADEMÁS S_medida (KVA) para los medidores que la traen real (45/50) -- mismo pipeline de 9 candidatos, GA y Optuna, sin distincion. Del histórico disponible de cada medidor, `CFG['train_frac']` (80%) cronológico entrena y el resto evalúa honestamente. Dentro del tramo de entrenamiento, las últimas `CFG['es_days']` se reservan para el *early stopping* de los modelos de boosting (nunca para ajustar pesos ni para elegir hiperparámetros). La selección se hace por CV, nunca por el test final. Evaluación rolling de un paso, diferenciada del pronóstico recursivo. Por cada medidor/variable se grafica la comparación de CV_RMSE entre todos los modelos y benchmarks evaluados, y ADEMÁS cada uno de los 6 modelos ML (no solo el ganador) obtiene su propia figura de dos paneles: convergencia/aprendizaje (error de validación por ronda de boosting en XGBoost/LightGBM/CatBoost, loss por iteración en MLP, o coeficientes/importancia de features en Ridge/RandomForest, que no son iterativos) y prueba real-vs-predicho sobre el tramo de evaluación honesta. Los 3 benchmarks Estacional_* no tienen figura individual (no hay nada que aprender ni converger en una regla de persistencia) pero sí quedan en la comparación de CV_RMSE y en las tablas de métricas/diagnósticos.


In [ ]:
# Umbral subido de 3500 (~36 dias) a 8640 (90 dias/12 semanas): con ventanas
# mas cortas casi no hay ciclos semanales completos para aprender el patron por
# dia de semana, ni suficiente historia para la señal 'mismo dia/hora, ultimas
# 4 semanas' (celda 12). Reduce cobertura mas no descarta datos: el medidor sigue
# disponible en la matriz historica y en figuras, solo no entra a la seleccion
# automatica de modelo/pronostico si tiene muy poca historia confiable.
from prodma.seleccion_features import seleccionar_features_ga
UMBRAL_ELEGIBILIDAD_P_VALIDOS=8640
usable=[m for m in MEDIDORES if any(x['medidor']==m and x['info']['P_validos']>UMBRAL_ELEGIBILIDAD_P_VALIDOS for x in INDEX)]
TARGETS=usable[:CFG['max_medidores_rapido']] if CFG['modo']=='rapido' else usable
RESULTS=[]; WINNERS={}; PRED={}

def graficar_modelo_individual(m,var,name,estimador,columnas,ye,pr):
    # Los 6 modelos ML quedan envueltos en TransformedTargetRegressor (celda 13,
    # correccion de heterocedasticidad): el estimador real esta en .regressor_.
    modelo_interno=getattr(estimador,'regressor_',estimador)
    try:
        fig,(ax_izq,ax_der)=plt.subplots(1,2,figsize=(12,5))
        curva=None;etiqueta_x=None
        if name=='XGBoost' and hasattr(modelo_interno,'evals_result'):
            res=modelo_interno.evals_result()
            if res:
                bloque=next(iter(res.values()));metric_name=next(iter(bloque.keys()))
                curva=bloque[metric_name];etiqueta_x=f'Ronda de boosting ({metric_name})'
        elif name=='LightGBM' and getattr(modelo_interno,'evals_result_',None):
            bloque=next(iter(modelo_interno.evals_result_.values()));metric_name=next(iter(bloque.keys()))
            curva=bloque[metric_name];etiqueta_x=f'Ronda de boosting ({metric_name})'
        elif name=='CatBoost' and hasattr(modelo_interno,'get_evals_result'):
            res=modelo_interno.get_evals_result()
            bloque=res.get('validation') or (next(iter(res.values())) if res else None)
            if bloque:
                metric_name=next(iter(bloque.keys()));curva=bloque[metric_name]
                etiqueta_x=f'Ronda de boosting ({metric_name})'
        elif name=='MLP':
            mlp=modelo_interno.named_steps.get('mlpregressor') if hasattr(modelo_interno,'named_steps') else None
            if mlp is not None and hasattr(mlp,'loss_curve_'):
                curva=mlp.loss_curve_;etiqueta_x='Iteracion (loss de entrenamiento)'
        if curva is not None:
            ax_izq.plot(curva,color='tab:red')
            ax_izq.set_xlabel(etiqueta_x);ax_izq.set_ylabel('error')
            ax_izq.set_title('Convergencia / aprendizaje')
        elif name=='Ridge':
            ridge=modelo_interno.named_steps.get('ridge') if hasattr(modelo_interno,'named_steps') else None
            if ridge is not None and hasattr(ridge,'coef_'):
                coefs=pd.Series(ridge.coef_,index=columnas).abs().sort_values(ascending=False).head(15)
                ax_izq.barh(coefs.index[::-1],coefs.to_numpy()[::-1],color='tab:blue')
                ax_izq.set_xlabel('|coeficiente| (estandarizado)')
                ax_izq.set_title('Que aprendio: coeficientes Ridge (top 15)')
            else:
                ax_izq.axis('off')
        elif name=='RandomForest' and hasattr(modelo_interno,'feature_importances_'):
            imp=pd.Series(modelo_interno.feature_importances_,index=columnas).sort_values(ascending=False).head(15)
            ax_izq.barh(imp.index[::-1],imp.to_numpy()[::-1],color='tab:green')
            ax_izq.set_xlabel('Importancia (impureza)')
            ax_izq.set_title('Que aprendio: importancia de features (top 15)')
        else:
            ax_izq.axis('off')
            ax_izq.text(0.5,0.5,'Sin curva de convergencia aplicable',ha='center',va='center',transform=ax_izq.transAxes)
        ye_np=np.asarray(ye,dtype=float);pr_np=np.asarray(pr,dtype=float)
        ax_der.scatter(ye_np,pr_np,s=8,alpha=0.35,color='tab:purple')
        finitos=np.isfinite(ye_np)&np.isfinite(pr_np)
        if finitos.any():
            lim=[min(ye_np[finitos].min(),pr_np[finitos].min()),max(ye_np[finitos].max(),pr_np[finitos].max())]
            ax_der.plot(lim,lim,'k--',linewidth=1)
        ax_der.set_xlabel('Real');ax_der.set_ylabel('Predicho')
        ax_der.set_title('Prueba: real vs. predicho (tramo de evaluacion)')
        fig.suptitle(f'{m} — {var}: {name}')
        fig.tight_layout()
        fig.savefig(RUN/f'figuras/modelo_{TOKENS[m]}_{var}_{name}.png',dpi=110)
        plt.close(fig)
    except Exception as exc:
        log('figura_modelo',f'{m}/{var}/{name}','ERROR',exc)


In [ ]:
assert not CFG.get('reanudar_run'),'Reanudación activa: el entrenamiento no debe ejecutarse '\
    '(use la CLI: python -m prodma.pipeline --reanudar <carpeta>)'
for m in TARGETS:
  d=load_meter(m)
  # S_medida entra al entrenamiento igual que P/Q cuando el medidor la trae real
  # (limpia en celda 10 con el mismo criterio); los 5 medidores sin KVA real solo
  # entrenan P y Q, como antes.
  tiene_s_real=('S_medida' in d.columns) and d['S_medida'].notna().any()
  variables_medidor=('P','Q')+(('S_medida',) if tiene_s_real else ())
  for var in variables_medidor:
    try:
      y=d[var].replace([np.inf,-np.inf],np.nan);last=y.last_valid_index()
      if last is None: raise ValueError('Sin datos observados')
      days=CFG['train_days_'+CFG['modo']]
      # days=None => usar todo el historico disponible del medidor hasta el corte
      # (modo 'completo': no recortar a una ventana mas corta que el dataset cargado)
      y=y.loc[:last] if days is None else y.loc[last-pd.Timedelta(days=days+CFG['test_days']+10):last]
      X=features(y);good=X.notna().all(axis=1)&y.notna()
      # Split por porcentaje de TODO el historico disponible (ya acotado arriba por
      # train_days_rapido/completo), no por una cantidad fija de dias: el 80% (train_frac)
      # cronologico mas antiguo de las filas utilizables (good) entrena, el resto evalua.
      pos=np.flatnonzero(good.to_numpy());n_good=len(pos)
      if n_good<CFG['min_train_days']*96+96: raise ValueError('Historial insuficiente')
      n_tr=int(round(n_good*CFG['train_frac']))
      n_tr=min(max(n_tr,CFG['min_train_days']*96),n_good-96)
      tr=np.zeros(len(X),dtype=bool);tr[pos[:n_tr]]=True
      te=np.zeros(len(X),dtype=bool);te[pos[n_tr:]]=True
      if tr.sum()<CFG['min_train_days']*96 or te.sum()<96: raise ValueError('Historial insuficiente')
      Xt_todo,yt_todo=X.loc[tr],y.loc[tr];Xe,ye=X.loc[te],y.loc[te]
      cv_todo=temporal_cv(Xt_todo.index,CFG['cv_folds'],CFG['cv_days'])
      # Seleccion de features por algoritmo genetico (una vez por serie, reutilizada
      # por los 6 modelos ML): con ventanas de entrenamiento limitadas, usar SIEMPRE
      # las columnas completas de features() puede generalizar peor que un
      # subconjunto mas chico. Los benchmarks Estacional_* siguen usando Xt_todo/Xe
      # completos (solo necesitan su columna de lag, no dependen de esta seleccion).
      columnas_ga,fitness_ga=seleccionar_features_ga(Xt_todo,yt_todo,cv_todo,
          poblacion=CFG['ga_poblacion'],generaciones=CFG['ga_generaciones'],semilla=CFG['seed'])
      print(f'{m} {var}: GA selecciono {len(columnas_ga)}/{X.shape[1]} features (RMSE CV={-fitness_ga:.3f})')
      Xe_ga=Xe[columnas_ga]
      scores=[]
      for name,(est,grid_params) in models().items():
        try:
          started=time.time()
          # Xt/Xval: dentro del 80% de entrenamiento, se separan las ultimas CFG['es_days']
          # para early stopping (XGBoost/LightGBM/CatBoost); el resto de modelos entrena
          # con Xt_todo completo. cv se recalcula sobre Xt (nunca incluye el tramo de ES).
          Xt,yt,Xval,yval=es_split(Xt_todo,yt_todo,name)
          Xt,Xval=Xt[columnas_ga],Xval[columnas_ga]
          cv=temporal_cv(Xt.index,CFG['cv_folds'],CFG['cv_days'])
          search=OptunaSearchCV(est,grid_params,cv=cv,scoring='neg_root_mean_squared_error',n_jobs=1,
                        n_trials=CFG['n_trials_optuna'],random_state=CFG['seed'],refit=True,
                        error_score='raise',verbose=0)
          search.fit(Xt,yt,**fit_kwargs(name,Xval,yval));pr=search.predict(Xe_ga)
          if not np.isfinite(pr).all():raise ValueError('Modelo generó valores no finitos')
          row=dict(medidor=m,variable=var,modelo=name,CV_RMSE=-search.best_score_,segundos=time.time()-started,
                   parametros=str(search.best_params_),**metrics(ye,pr))
          RESULTS.append(row);scores.append((row['CV_RMSE'],name,search.best_estimator_))
          PRED[(m,var,name)]=pd.DataFrame({'real':ye,'pred':pr})
          joblib.dump(search.best_estimator_,RUN/'modelos'/f'{TOKENS[m]}_{var}_{name}.joblib')
          # Figura PROPIA de este modelo (no solo del ganador): convergencia/aprendizaje
          # cuando el modelo es iterativo, o que aprendio via coeficientes/importancia
          # cuando no lo es (Ridge/RandomForest), mas el panel de prueba real-vs-predicho.
          graficar_modelo_individual(m,var,name,search.best_estimator_,columnas_ga,ye,pr)
        except Exception as exc: log('modelo',f'{m}/{var}/{name}','ERROR',exc)
      # Benchmarks puntuados también en ventanas de CV (sobre Xt_todo/cv_todo: el
      # conjunto de entrenamiento completo, no el recorte de early stopping de un
      # modelo particular; cv_todo ya se calculo arriba, antes de la seleccion GA).
      for lag in (1,96,672):
        name=f'Estacional_{lag}';pred=Xe[f'lag_{lag}'];cv_scores=[]
        for _,vi in cv_todo: cv_scores.append(metrics(yt_todo.iloc[vi],Xt_todo.iloc[vi][f'lag_{lag}'])['RMSE'])
        score=float(np.mean(cv_scores));scores.append((score,name,lag))
        RESULTS.append(dict(medidor=m,variable=var,modelo=name,CV_RMSE=score,**metrics(ye,pred)))
        PRED[(m,var,name)]=pd.DataFrame({'real':ye,'pred':pred})
      # El ganador OPERATIVO es siempre un modelo ML: los benchmarks Estacional_* (modelo=int)
      # se puntúan aparte, pero en recursión a meses/años 'Estacional_1' es una línea plana.
      score,name,model=min((s for s in scores if not isinstance(s[2],(int,np.integer))),key=lambda x:x[0])
      WINNERS[(m,var)]={'name':name,'model':model,'cv_rmse':score,'last':last,'columns':columnas_ga,
                       'test_inicio':Xe.index[0]}
      print(m,var,'seleccionado:',name)
      pd.DataFrame(RESULTS).to_csv(RUN/'tablas/metricas.csv',index=False)
      # Gráfica comparativa de métricas (CV_RMSE) entre TODOS los modelos/benchmarks
      # evaluados para este medidor/variable; el ganador queda resaltado.
      try:
        dfm=pd.DataFrame(RESULTS);sub=dfm[(dfm.medidor==m)&(dfm.variable==var)].sort_values('CV_RMSE')
        if len(sub):
          fig,ax=plt.subplots(figsize=(8,5))
          colors=['tab:orange' if n==name else 'tab:gray' for n in sub.modelo]
          ax.bar(sub.modelo,sub.CV_RMSE,color=colors);ax.tick_params(axis='x',rotation=45)
          ax.set_ylabel('CV RMSE');ax.set_title(f'{m} — {var}: comparación de modelos (ganador: {name})')
          fig.tight_layout();fig.savefig(RUN/f'figuras/metricas_{TOKENS[m]}_{var}.png',dpi=120);plt.close(fig)
      except Exception as exc: log('figura_metricas',f'{m}/{var}','ERROR',exc)
    except Exception as exc: log('entrenamiento',f'{m}/{var}','NO_ELEGIBLE',exc)
assert WINNERS,'No se entrenó ninguna serie; revisar logs y configuración'

## 14b · Restauración del estado (solo al reanudar)

Con `CFG['reanudar_run']` (CLI: `--reanudar <carpeta>`) las secciones 14–16 no se ejecutan: sus resultados ya están en la corrida indicada. Esta celda reconstruye `PRED`/`WINNERS` desde los `.joblib` y `metricas.csv`, **valida** que cada modelo guardado reproduzca su RMSE guardado y clasifica cada serie (`OK`, `SERIE_CONSTANTE`, `SIN_DATOS_RECIENTES`). El ganador operativo es el mejor modelo **ML** por `CV_RMSE`: los benchmarks `Estacional_*` quedan en las tablas, pero no se usan como pronóstico (recursivamente, `Estacional_1` es una línea plana).

In [ ]:
# --- 14b · Restauración del estado desde una corrida previa (solo con CFG['reanudar_run']) ---
# Reemplaza a las secciones 14-16 al reanudar: sus resultados ya están en disco
# (modelos/*.joblib, tablas/metricas.csv, comparacion_completa.csv, OLS_referencia.csv).
# NO se entrena nada: se reconstruyen PRED/WINNERS y se VALIDA que cada modelo guardado
# reproduzca su RMSE guardado (si no, se aborta: la restauración no sería fiel).
from prodma.reanudar import (restaurar_estado,construir_ganadores,guardar_estado,cargar_estado,
    ESTADO_OK,ESTADO_CONSTANTE,ESTADO_SIN_DATOS)
if not CFG.get('reanudar_run'):
    print('Sin reanudación: el estado (PRED/WINNERS) proviene del entrenamiento de la sección 14.')
else:
    _previo=cargar_estado(RUN)
    if _previo is None:
        _estado=restaurar_estado(RUN,cfg=CFG,tokens=TOKENS,load_meter=load_meter,features=features,metrics=metrics)
        _estado['validacion'].to_csv(RUN/'tablas/validacion_restauracion.csv',index=False)
        WINNERS=construir_ganadores(pd.DataFrame(_estado['results']),_estado,RUN,TOKENS,
                                    max_dias_sin_datos=CFG['max_dias_sin_datos'])
        _ruta=guardar_estado(RUN,_estado,WINNERS)   # checkpoint: las próximas reanudaciones tardan segundos
        print('Estado restaurado y guardado en',_ruta)
    else:
        _estado,WINNERS=_previo
        print('Estado restaurado desde el checkpoint',RUN/'tablas'/'estado_restaurado.joblib')
    RESULTS=_estado['results'];PRED=_estado['pred']
    _v=_estado['validacion']
    print(f"Validación: {len(_v)} modelos/benchmarks reproducidos; dif. relativa máx. de RMSE = {_v.dif_rel.max():.2e}")
    ESTADO_SERIES=pd.DataFrame([dict(medidor=m,variable=v,modelo_operativo=w['name'],cv_rmse=w['cv_rmse'],
        ultimo_dato=w['last'],estado=w['estado'],motivo=w['motivo']) for (m,v),w in WINNERS.items()])
    ESTADO_SERIES.to_csv(RUN/'tablas/estado_series.csv',index=False)
    print(ESTADO_SERIES.estado.value_counts().to_string())
    for (m,v),w in WINNERS.items():
        if w['estado']!=ESTADO_OK: log('clasificacion_series',f'{m}/{v}',w['estado'],w['motivo'])


## 15 · Diagnósticos de todos los modelos

Los p-valores con dependencia son aproximados. Severidad no se deduce solo del p-valor; ACF y magnitud se reportan.

**Criterios mínimos de aceptación** (se calculan para los 6 modelos y los 3 benchmarks estacionales; se reportan en `comparacion_completa.csv` y se registran como ADVERTENCIA si no se cumplen — nunca descartan un modelo ni detienen la ejecución de otros medidores):

- **Durbin-Watson** dentro de `CFG['rango_durbin_watson']` (sin autocorrelación de primer orden en los residuos).
- **Ljung-Box** con p ≥ `CFG['alfa_significancia']` (sin autocorrelación residual a varios rezagos).
- **Breusch-Pagan y White** — aquí calculados sobre el **valor predicho** (no sobre los regresores originales, porque estos modelos no son lineales) — con p ≥ `CFG['alfa_significancia']` (homocedasticidad: la varianza del error no depende del nivel predicho).
- **R² fuera de muestra** ≥ `CFG['r2_minimo_aceptable']`.

**F, R² ajustado y la significancia t-Student de cada coeficiente NO aplican** a modelos no lineales (RandomForest, XGBoost, LightGBM, CatBoost, MLP) ni a Ridge regularizado: son estadística inferencial de regresión lineal clásica y se calculan aparte, sobre un OLS de referencia, en la sección 16.



In [ ]:
from statsmodels.stats.diagnostic import acorr_ljungbox, het_breuschpagan, het_white
from statsmodels.stats.stattools import durbin_watson, jarque_bera
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from scipy.stats import probplot
import statsmodels.api as sm
ALFA=CFG['alfa_significancia'];RANGO_DW=CFG['rango_durbin_watson'];UMBRAL_R2_MINIMO=CFG['r2_minimo_aceptable']
DIAG=[]
for (m,var,name),pred in PRED.items():
    e=(pred.real-pred.pred).dropna();row=dict(medidor=m,variable=var,modelo=name,R2_adj=np.nan,F=np.nan,
       F_p=np.nan,BP_p=np.nan,White_p=np.nan,BP_pred_p=np.nan,White_pred_p=np.nan,
       aplicabilidad='F, R2_adj y significancia t-Student de coeficientes: N/A para modelo predictivo no lineal (ver OLS_referencia.csv, seccion 16, para inferencia clasica)')
    try:
      row['DW']=durbin_watson(e); jb,jp,sk,ku=jarque_bera(e)
      row.update(JB=jb,JB_p=jp,asimetria=sk,curtosis=ku,ACF1=e.autocorr(1))
      # No comprimir huecos y fingir que continúan siendo cuartos de hora.
      segments=(e.index.to_series().diff()!=pd.Timedelta(minutes=15)).cumsum()
      longest=max((v for _,v in e.groupby(segments)),key=len)
      lags=[v for v in (4,12,24,48,96,672) if v<len(longest)//5]
      if lags:
        lb=acorr_ljungbox(longest,lags=lags,return_df=True)
        row['LB_min_p']=lb.lb_pvalue.min();row['LB_n']=len(longest)
        row['alerta_autocorrelacion']=bool(row['LB_min_p']<ALFA)
        lb.to_csv(RUN/'tablas'/f'LB_{TOKENS[m]}_{var}_{name}.csv')
      for period in (96,672): row[f'ACF{period}']=longest.autocorr(period) if len(longest)>period else np.nan
      # Heterocedasticidad model-agnóstica: BP/White clásicos exigen los regresores del
      # modelo lineal (solo existen para el OLS de la sección 16). Aquí se prueba si la
      # varianza del error depende del NIVEL PREDICHO -- válido para cualquier modelo,
      # incluidos árboles y redes: es la misma lectura del gráfico predicho-vs-residuo de
      # la sección 17, con un p-valor formal en lugar de solo visual.
      pred_alineado=pred.pred.reindex(e.index)
      ok_bp=pred_alineado.notna()&np.isfinite(pred_alineado)&np.isfinite(e)
      if ok_bp.sum()>=30:
        exog_pred=sm.add_constant(pred_alineado[ok_bp].to_numpy())
        row['BP_pred_p']=het_breuschpagan(e[ok_bp].to_numpy(),exog_pred)[1]
        row['White_pred_p']=het_white(e[ok_bp].to_numpy(),exog_pred)[1]
    except Exception as exc: row['error']=str(exc)
    DIAG.append(row)

diagnosticos=pd.DataFrame(DIAG)
diagnosticos.to_csv(RUN/'tablas/diagnostico_modelos.csv',index=False)

comparacion=pd.DataFrame(RESULTS).merge(diagnosticos,on=['medidor','variable','modelo'],how='left')
# Batería mínima de aceptación (ver comentario superior); cada columna queda en NaN cuando
# el criterio no pudo evaluarse (no se confunde "no evaluado" con "no cumple").
comparacion['cumple_DW']=comparacion['DW'].between(*RANGO_DW).where(comparacion['DW'].notna())
comparacion['cumple_no_autocorrelacion']=(comparacion['LB_min_p']>=ALFA).where(comparacion['LB_min_p'].notna())
comparacion['cumple_homocedasticidad_BP']=(comparacion['BP_pred_p']>=ALFA).where(comparacion['BP_pred_p'].notna())
comparacion['cumple_homocedasticidad_White']=(comparacion['White_pred_p']>=ALFA).where(comparacion['White_pred_p'].notna())
comparacion['cumple_R2_0_90']=(comparacion['R2']>=UMBRAL_R2_MINIMO).where(comparacion['R2'].notna())
CRITERIOS_MINIMOS=['cumple_DW','cumple_no_autocorrelacion','cumple_homocedasticidad_BP','cumple_homocedasticidad_White','cumple_R2_0_90']
n_evaluados=comparacion[CRITERIOS_MINIMOS].notna().sum(axis=1)
comparacion['cumple_minimos']=comparacion[CRITERIOS_MINIMOS].fillna(True).all(axis=1).astype(object)
comparacion.loc[n_evaluados==0,'cumple_minimos']=np.nan
comparacion.to_csv(RUN/'tablas/comparacion_completa.csv',index=False)

# Advertencia explícita (no detiene la ejecución: un medidor no se queda sin pronóstico
# solo porque su modelo ganador no supere todos los umbrales sobre datos reales de campo).
for (m,var),w in WINNERS.items():
    fila=comparacion[(comparacion.medidor==m)&(comparacion.variable==var)&(comparacion.modelo==w['name'])]
    if len(fila):
        valor=fila['cumple_minimos'].iloc[0]
        if pd.notna(valor) and not bool(valor):
            incumplidos=[c for c in CRITERIOS_MINIMOS if pd.notna(fila[c].iloc[0]) and not bool(fila[c].iloc[0])]
            log('criterios_estadisticos',f'{m}/{var}/{w["name"]}','ADVERTENCIA',
                'No cumple: '+', '.join(incumplidos)+' (ver comparacion_completa.csv)')
print('Diagnósticos y criterios mínimos de aceptación calculados para',len(diagnosticos),'combinaciones medidor/variable/modelo')


In [ ]:
# --- Seleccion estadistica estricta: entre los candidatos de esta serie, preferir el
# de menor CV_RMSE que SI cumple TODOS los criterios minimos (Durbin-Watson, Ljung-Box,
# homocedasticidad BP/White, R2>=umbral) en vez de aceptar el de menor error sin mirar
# sus diagnosticos. Si ningun candidato cumple, se conserva el ganador original (ya
# quedo la ADVERTENCIA registrada arriba) -- un medidor no se queda sin pronostico solo
# porque ningun candidato paso los minimos sobre datos reales de campo.
n_reemplazados=0
for (m,var),w in list(WINNERS.items()):
    candidatos=comparacion[(comparacion.medidor==m)&(comparacion.variable==var)].sort_values('CV_RMSE')
    aprobados=candidatos[(candidatos['cumple_minimos']==True)&(~candidatos['modelo'].str.startswith('Estacional_'))]
    if len(aprobados)==0 or w['name'] in aprobados['modelo'].to_numpy():
        continue
    nombre_nuevo=aprobados.iloc[0]['modelo']
    if nombre_nuevo.startswith('Estacional_'):
        modelo_nuevo=int(nombre_nuevo.split('_')[1])
    else:
        ruta=RUN/'modelos'/f'{TOKENS[m]}_{var}_{nombre_nuevo}.joblib'
        modelo_nuevo=joblib.load(ruta) if ruta.exists() else None
    if modelo_nuevo is None: continue
    WINNERS[(m,var)]={'name':nombre_nuevo,'model':modelo_nuevo,
                       'cv_rmse':float(aprobados.iloc[0]['CV_RMSE']),'last':w['last'],'columns':w['columns']}
    n_reemplazados+=1
    log('seleccion_estricta',f'{m}/{var}','INFO',
        f'Ganador original ({w["name"]}) no cumplia los criterios minimos; reemplazado por {nombre_nuevo}, que si los cumple.')
    print(f'{m} {var}: reemplazado por seleccion estadistica estricta -> {nombre_nuevo}')
print(f'Seleccion estadistica estricta: {n_reemplazados} ganador(es) reemplazados de {len(WINNERS)} series.')


## 16 · OLS inferencial independiente

OLS sobre entrenamiento: F clásica, BP/White y covarianza HAC. No se atribuyen sus resultados a Ridge ni ML.

**Criterios mínimos de aceptación de esta sección** (única con inferencia lineal clásica válida, sobre un OLS de 5 variables independiente del ganador ML): estadístico **F** con p < `CFG['alfa_significancia']`; **todos** los coeficientes con **t-Student** de p < `CFG['alfa_significancia']`; **R² ajustado** ≥ `CFG['r2_minimo_aceptable']`; **Durbin-Watson** dentro de `CFG['rango_durbin_watson']`; **Breusch-Pagan y White** con p ≥ `CFG['alfa_significancia']`. El incumplimiento se registra como ADVERTENCIA en el log y queda en la columna `cumple_minimos_ols` de `OLS_referencia.csv`; nunca se oculta ni se fuerza a pasar.



In [ ]:
import statsmodels.api as sm
from statsmodels.stats.stattools import durbin_watson
# Única sección donde F, R² ajustado y la significancia t-Student de CADA coeficiente
# tienen sentido estadístico: exigen un modelo lineal con los supuestos de OLS, que
# Ridge/RandomForest/XGBoost/LightGBM/CatBoost/MLP no cumplen (ver sección 15).
OLS=[]
for m,var in WINNERS:
  try:
    y=load_meter(m)[var];X=features(y)[['hora_sin1','hora_cos1','dia_1','dia_2','dia_3','dia_4','dia_5','dia_6','lag_96']]
    cutoff=y.last_valid_index()-pd.Timedelta(days=CFG['test_days'])
    data=X.join(y.rename('y')).loc[:cutoff].dropna().tail(96*365)
    ex=sm.add_constant(data.drop(columns='y'));fit=sm.OLS(data.y,ex).fit()
    bp=het_breuschpagan(fit.resid,ex);white=het_white(fit.resid,ex)
    hac=fit.get_robustcov_results(cov_type='HAC',maxlags=96)
    dw=durbin_watson(fit.resid)
    todos_coef_significativos=bool((fit.pvalues<CFG['alfa_significancia']).all())
    fila=dict(medidor=m,variable=var,n=len(data),R2_adj=fit.rsquared_adj,F=fit.fvalue,F_p=fit.f_pvalue,
              BP_p=bp[1],White_p=white[1],HAC_F_p=hac.f_pvalue,DW=dw,
              coeficientes_t=str(fit.tvalues.to_dict()),coeficientes_p=str(fit.pvalues.to_dict()),
              todos_coeficientes_significativos=todos_coef_significativos)
    fila['cumple_F']=bool(fit.f_pvalue<CFG['alfa_significancia'])
    fila['cumple_R2_adj_0_90']=bool(fit.rsquared_adj>=CFG['r2_minimo_aceptable'])
    fila['cumple_homocedasticidad_BP']=bool(bp[1]>=CFG['alfa_significancia'])
    fila['cumple_homocedasticidad_White']=bool(white[1]>=CFG['alfa_significancia'])
    fila['cumple_DW']=bool(CFG['rango_durbin_watson'][0]<=dw<=CFG['rango_durbin_watson'][1])
    fila['cumple_minimos_ols']=all([fila['cumple_F'],fila['cumple_R2_adj_0_90'],
        fila['cumple_homocedasticidad_BP'],fila['cumple_homocedasticidad_White'],
        fila['cumple_DW'],todos_coef_significativos])
    OLS.append(fila)
    if not fila['cumple_minimos_ols']:
        incumplidos=[k for k in ('cumple_F','cumple_R2_adj_0_90','cumple_homocedasticidad_BP',
            'cumple_homocedasticidad_White','cumple_DW') if not fila[k]]
        if not todos_coef_significativos: incumplidos.append('coeficientes_no_significativos')
        log('criterios_estadisticos_ols',f'{m}/{var}','ADVERTENCIA',
            'OLS de referencia no cumple: '+', '.join(incumplidos)+' (ver OLS_referencia.csv)')
    (RUN/'tablas'/f'OLS_{TOKENS[m]}_{var}.txt').write_text(str(fit.summary())+'\nHAC:\n'+str(hac.summary()))
  except Exception as exc: log('OLS',f'{m}/{var}','ERROR',exc)
pd.DataFrame(OLS).to_csv(RUN/'tablas/OLS_referencia.csv',index=False)
print('OLS de referencia (inferencia clásica) calculado para',len(OLS),'series; ver cumple_minimos_ols en OLS_referencia.csv')


## 17 · Figuras por modelo

Real/predicho, residuos, dispersión, histograma, Q–Q, ACF/PACF y error por hora.


In [ ]:
import gc,psutil,matplotlib
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from scipy.stats import probplot
# Backend sin GUI durante las ~1300 figuras (con 'inline' cada figura se registra además en
# el kernel) y liberación explícita de memoria: la versión anterior terminó con MemoryError
# tras ~890 figuras. Las figuras que ya existen en disco (p. ej. de una corrida interrumpida)
# se omiten, así que esta celda se puede volver a ejecutar hasta completar.
_backend_previo=matplotlib.get_backend();plt.switch_backend('Agg')
_proceso=psutil.Process();_nuevas=_omitidas=0
try:
    for (m,var,name),d in PRED.items():
        destino=RUN/'figuras'/f'{TOKENS[m]}_{var}_{name}.png'
        if destino.exists(): _omitidas+=1;continue
        e=d.real-d.pred;fig,ax=plt.subplots(4,2,figsize=(15,14))
        d.tail(672).plot(ax=ax[0,0]);ax[0,1].plot(e.index,e)
        ax[1,0].scatter(d.pred,e,s=2);ax[1,1].hist(e,bins=40)
        probplot(e,plot=ax[2,0])
        if e.std()>1e-10 and len(e)>100:
            plot_acf(e,lags=min(96,len(e)//5),ax=ax[2,1])
            plot_pacf(e,lags=min(48,len(e)//5),ax=ax[3,0],method='ywm')
        else:
            ax[2,1].text(.1,.5,'ACF N/A: residuos constantes o insuficientes')
            ax[3,0].text(.1,.5,'PACF N/A')
        e.abs().groupby(e.index.hour).mean().plot(ax=ax[3,1])
        fig.suptitle(f'{m} — {var} ({"kW" if var=="P" else "kvar"}) — {name}; test observado')
        fig.tight_layout();fig.savefig(destino,dpi=120);plt.close(fig);_nuevas+=1
        if _nuevas%50==0:
            gc.collect()
            print(f'  {_nuevas} figuras nuevas ({_omitidas} ya existían); RSS del kernel {_proceso.memory_info().rss/2**20:.0f} MB',flush=True)
finally:
    plt.close('all');plt.switch_backend(_backend_previo)
print(f'Figuras de diagnóstico: {_nuevas} nuevas, {_omitidas} ya existentes; RSS final {_proceso.memory_info().rss/2**20:.0f} MB')


## 18 · Clientes: lectura real

Reconoce los encabezados originales con errores ortográficos y las columnas anuales. No fuerza coherencia donde no existe.


In [ ]:
CLIENTS=[];CLIENT_AUDIT=[]
if not CLIENTES_DISPONIBLE:
    log('clientes','archivo_nuevosclientes','PENDIENTE','No encontrado en la carpeta local; sin nuevas cargas en esta ejecución')
    print('Archivo de nuevos clientes no disponible: se continúa sin altas (CLIENTS=[]); no se inventan clientes.')
else:
    # Motor de lectura por extensión real (el archivo puede llegar como .xls o .xlsx).
    ext_clientes=Path(paths[CLIENTE_ARCHIVO_KEY]).suffix.lower()
    engine_clientes='openpyxl' if ext_clientes=='.xlsx' else 'xlrd'
    book=pd.ExcelFile(paths[CLIENTE_ARCHIVO_KEY],engine=engine_clientes)
    for sn in book.sheet_names:
        raw=pd.read_excel(book,sheet_name=sn,header=None)
        hr=None
        for i in range(min(30,len(raw))):
            if 'demanda kw' in [norm(x) for x in raw.iloc[i]]: hr=i;break
        if hr is None:
            log('clientes',sn,'OMITIDA','Sin encabezado demanda kw');continue
        cols_raw=[str(x).strip() for x in raw.iloc[hr]]
        df=raw.iloc[hr+1:].copy();df.columns=cols_raw
        # Encabezados vacíos (celda en blanco -> 'nan'/''): se descartan; no son columnas
        # reales y no deben tratarse como ambigüedad. Solo un nombre real repetido es error.
        es_vacio=lambda c:c=='' or c.strip().lower() in ('nan','none')
        col_valida=[not es_vacio(c) for c in cols_raw]
        if not all(col_valida):
            log('clientes',sn,'INFO',f'{sum(1 for v in col_valida if not v)} columna(s) con encabezado vacío descartada(s)')
        df=df.loc[:,col_valida]
        if df.columns.duplicated().any():
            dup=[c for c in df.columns[df.columns.duplicated()]]
            raise ValueError(f'{sn}: columnas duplicadas en encabezado {dup}; no se puede mapear sin ambigüedad')
        maps={norm(c):c for c in df.columns}
        client_col=maps.get('nombre del cleinet nuevo') or maps.get('nombre del cliente nuevo')
        medidor_col=maps.get('medidor')
        alimentador_col=maps.get('aliementador') or maps.get('alimentador')
        fp_col=maps.get('factor de potencia')
        faltan=[etiqueta for etiqueta,c in [('cliente',client_col),('demanda kw',maps.get('demanda kw')),
                ('medidor',medidor_col),('alimentador',alimentador_col),('factor de potencia',fp_col)] if not c]
        if faltan:
            log('clientes',sn,'OMITIDA',f'Columnas no encontradas: {faltan}');continue
        years=[c for c in df.columns if re.fullmatch(r'20\d{2}(\.0)?',c)]
        for _,r in df.iterrows():
            if pd.isna(r[client_col]):continue
            client=str(r[client_col]).strip();total=pd.to_numeric(r[maps['demanda kw']],errors='coerce')
            stages={int(float(c)):pd.to_numeric(r[c],errors='coerce') for c in years}
            schedule=sum(v for v in stages.values() if pd.notna(v))
            CLIENT_AUDIT.append(dict(cliente=client,solicitado=total,suma_etapas=schedule,diferencia=schedule-total))
            prev=0
            for year,val in sorted(stages.items()):
                if pd.isna(val):continue
                kw=val-prev if CFG['calendario_etapas']=='acumulado' else val
                if CFG['calendario_etapas']=='acumulado':prev=val
                if kw==0:continue
                cid=f'{client}|{year}';adj=AJUSTES_CLIENTES.get(cid,{})
                date=adj.get('fecha')
                if not date and CFG['mes_ingreso'] and CFG['dia_ingreso']:
                    date=f"{year}-{CFG['mes_ingreso']:02d}-{CFG['dia_ingreso']:02d}"
                CLIENTS.append(dict(id=cid,cliente=client,year=year,kw=adj.get('kw',kw),
                    medidor=adj.get('medidor',str(r[medidor_col]).strip()),
                    alimentador=str(r[alimentador_col]).strip(),fp=float(r[fp_col]),
                    fecha=date,perfil='constante_15min',rampa_dias=0,
                    incluido_hist=adj.get('incluido_hist'),confirmado=adj.get('confirmado',False)))
clients=pd.DataFrame(CLIENTS);client_audit=pd.DataFrame(CLIENT_AUDIT)
clients.to_csv(RUN/'tablas/clientes_etapas_pendientes.csv',index=False)
client_audit.to_csv(RUN/'tablas/clientes_coherencia.csv',index=False)
display(client_audit);display(clients)
print('Revise AJUSTES_CLIENTES. La demanda solicitada NO se suma además de sus etapas.')

## 19 · Altas constantes por Medidor

Cada etapa confirmada se suma completa en todos los cuartos de hora desde su fecha. Sin tasas, rampas, perfiles horarios ni factores reductores. Medidor es la clave exacta; alimentador es informativo.


In [ ]:
def customer_profile(index,c):
    if not c['confirmado']:raise ValueError('Etapa sin confirmar')
    if not c.get('fecha'):raise ValueError('Falta fecha exacta o supuesto declarado')
    fp=float(c['fp'])
    if not 0<fp<=1:raise ValueError('Factor de potencia inválido')
    start=pd.Timestamp(c['fecha'])
    if pd.isna(start):raise ValueError('Fecha inválida')
    kw=float(c['kw'])
    if not np.isfinite(kw):raise ValueError('Demanda no finita')
    p=kw*np.asarray(index>=start,dtype=float);q=p*np.tan(np.arccos(fp))
    return p,q
def add_new_customers(index,medidor,records):
    out=pd.DataFrame({'P_nuevos':0.,'Q_nuevos':0.},index=index);pending=[];seen=set()
    for c in records:
        # Comparación exacta del código fuente, sin aproximación ni sustitución por alimentador.
        if c['medidor']!=medidor:continue
        if c['id'] in seen:raise ValueError('Etapa de cliente duplicada: '+c['id'])
        seen.add(c['id'])
        if c['incluido_hist'] is True:continue
        try:
            if c['incluido_hist'] is not False:raise ValueError('Confirmar que no está incluido en histórico')
            p,q=customer_profile(index,c)
            out['P_nuevos']+=p;out['Q_nuevos']+=q
        except (ValueError,TypeError) as exc:pending.append({'id':c['id'],'motivo':str(exc)})
    return out,pending
check={'confirmado':True,'fecha':'2027-01-02','perfil':'continuo','fp':.97,'kw':100,'rampa_dias':0}
ii=pd.date_range('2027-01-01',periods=193,freq='15min');pp,qq=customer_profile(ii,check)
assert (pp[:96]==0).all() and (pp[96:]==100).all()
assert np.allclose(np.hypot(pp[96:],qq[96:]),100/.97)
print('Perfil, fecha y potencia reactiva verificados')
unmatched=clients.loc[~clients.medidor.isin(MEDIDORES)]
unmatched.to_csv(RUN/'tablas/clientes_medidor_sin_coincidencia.csv',index=False)
if len(unmatched):
    display(unmatched)
    log('clientes','Medidor','PENDIENTE','Códigos sin histórico; no se reasignan a otro alimentador')

## 20 · Proyección operativa hasta el cierre del horizonte

Aplica a P, Q y -- para los medidores con KVA real -- también a S_medida. **Sin reentrenar**: se usan los modelos guardados (entrenados con el 80% cronológico del histórico). El pronóstico recursivo de un modelo entrenado a 1 paso **pierde el nivel en pocos días** (la recursión converge a un punto fijo; el R² a 1 paso no lo revela), así que el método de cada serie se **elige por backtest a 28 días** sobre datos que el modelo no vio: ML recursivo, perfil calendario (mediana de 12 meses) o perfil anclado (forma estacional × nivel reciente desestacionalizado). El ML solo se usa donde mejora al mejor perfil en más de `1-CFG['margen_ml']` y, además, pasa las guardas de sanidad (media proyectada frente a los últimos 12 meses, aplanamiento, rango); si no, se usa el mejor perfil y queda registrado. Cada serie deja `fuente`, `pred_ml`, `pred_perfil_anclado` y `pred_perfil_calendario` en `pronosticos/serie_*.parquet`. Las series sin datos recientes no se proyectan y las constantes se proyectan como constante declarada. El cálculo recursivo usa `prodma.recursivo` (calendario vectorizado + numpy, validado en vivo contra `features()`), un proceso por serie y un checkpoint por serie. Las bandas son empíricas del backtest a 28 días y exploratorias: no hay validación a 16 meses. Por cada medidor/variable se grafica la curva mensual y la variación mensual (%).

In [ ]:
import matplotlib
from prodma.recursivo import (features_calendario,proyectar_series,proyectar_recursivo,
    preparar_para_inferencia,cargar_modelo,es_columna_dinamica,BUFFER_MINIMO)
from prodma.sanidad import evaluar_sanidad,calendar_profile,perfil_anclado
from prodma.reanudar import clasificar_serie,ESTADO_OK,ESTADO_CONSTANTE,ESTADO_SIN_DATOS
BUFFER_HISTORIA=BUFFER_MINIMO+96   # 4 semanas + 1 día: alcance máximo de las features dinámicas
FIN_PLAN=pd.Timestamp(CFG['fin_plan'])
H_SEL=max(CFG['backtest_horizontes'])   # horizonte (pasos de 15 min) con el que se elige el método de cada serie
FORECASTS={};MENSUAL={};SANIDAD=[]

# --- Estado de cada serie: solo las ACTIVAS entran a la elección de método. Las constantes se
# proyectan como constante declarada y las que dejaron de reportar NO se proyectan (proyectar
# años sobre un medidor apagado sería inventar demanda). ---
_ref_ultimo=max(w['last'] for w in WINNERS.values())
for (m,var),w in WINNERS.items():
    if 'estado' not in w: w['estado'],w['motivo']=clasificar_serie(w['cv_rmse'],w['last'],_ref_ultimo,CFG['max_dias_sin_datos'])
    if not w.get('ruta_modelo'): w['ruta_modelo']=str(RUN/'modelos'/f"{TOKENS[m]}_{var}_{w['name']}.joblib")
ACTIVAS=[k for k,w in WINNERS.items() if w['estado']==ESTADO_OK]
CONSTANTES=[k for k,w in WINNERS.items() if w['estado']==ESTADO_CONSTANTE]
SIN_DATOS=[k for k,w in WINNERS.items() if w['estado']==ESTADO_SIN_DATOS]
for k in SIN_DATOS: log('pronostico',f'{k[0]}/{k[1]}',ESTADO_SIN_DATOS,WINNERS[k]['motivo'])
for k in CONSTANTES: log('pronostico',f'{k[0]}/{k[1]}',ESTADO_CONSTANTE,'Se proyecta la mediana de los últimos 30 días (sin ML)')
print(f'Series: {len(ACTIVAS)} activas, {len(CONSTANTES)} constantes, {len(SIN_DATOS)} sin datos recientes (no se proyectan)')

def historia_recursion(m,var,hasta):
    hist,_=causal_fill(load_meter(m)[var].loc[:hasta]);return hist
def _perfiles(hist,idx):
    return {'PERFIL_ANCLADO':perfil_anclado(hist,idx),'PERFIL_CALENDARIO':calendar_profile(hist,idx)}
_desde_global=min(WINNERS[k]['last'] for k in ACTIVAS+CONSTANTES)+pd.Timedelta(minutes=15)
# Calendario/exógenas de TODO el horizonte, vectorizado y una sola vez (no dependen de la recursión).
CAL_GLOBAL=features_calendario(pd.date_range(_desde_global,FIN_PLAN,freq='15min',name='Fecha'),
    FERIADOS,CLIMATOLOGIA_TEMPERATURA,PBI_MENSUAL,POBLACION_ANUAL)

# --- Validación en vivo contra features() del notebook: el calendario vectorizado y la
# recursión rápida deben reproducir EXACTAMENTE lo que daría features() paso a paso. ---
_k0=next((k for k in ACTIVAS if WINNERS[k]['name'] in ('Ridge','LightGBM','XGBoost','CatBoost')),ACTIVAS[0])
_w0=WINNERS[_k0];_hist0=historia_recursion(*_k0,_w0['last'])
_Xref=features(load_meter(_k0[0])[_k0[1]].tail(4000))
_cal_ref=features_calendario(_Xref.index,FERIADOS,CLIMATOLOGIA_TEMPERATURA,PBI_MENSUAL,POBLACION_ANUAL)
for _c in _cal_ref.columns:
    assert np.allclose(_cal_ref[_c].to_numpy(float),_Xref[_c].to_numpy(float),atol=1e-9,equal_nan=True),f'calendario difiere de features(): {_c}'
_m0=cargar_modelo(_w0['ruta_modelo']);preparar_para_inferencia(_m0)
_idx0=pd.date_range(_w0['last']+pd.Timedelta(minutes=15),periods=24,freq='15min')
_s=_hist0.tail(BUFFER_HISTORIA).copy();_ref=[]
for _t in _idx0:
    _s.loc[_t]=np.nan;_v=float(_m0.predict(features(_s).loc[[_t],_w0['columns']])[0]);_s.loc[_t]=_v;_ref.append(_v);_s=_s.tail(BUFFER_HISTORIA)
_rap=proyectar_recursivo(_m0,_hist0.to_numpy()[-BUFFER_HISTORIA:],_w0['columns'],CAL_GLOBAL.loc[_idx0[0]:_idx0[-1]])
assert np.allclose(_ref,_rap,rtol=1e-6,atol=1e-6),'La recursión rápida no reproduce features() del notebook'
print(f'Validación: calendario y recursión rápida reproducen features() ({_k0[0]}/{_k0[1]}, {_w0["name"]}, diferencia máx. {np.abs(np.array(_ref)-_rap).max():.2e})')
del _m0,_Xref,_cal_ref,_s

# --- Backtest multi-paso SIN reentrenar y elección del método de cada serie -----------------------
# El pronóstico recursivo de un modelo entrenado a 1 paso pierde el nivel en pocos días (la
# recursión converge a un punto fijo); el R² a 1 paso no lo revela. Por eso el método de cada
# serie se elige por su error a H_SEL pasos (28 días) sobre los últimos test_days+1 días del tramo
# de prueba, que el modelo guardado nunca vio: ML recursivo, perfil calendario (mediana de 12
# meses) o perfil anclado (forma estacional x nivel reciente). El ML solo gana si mejora al mejor
# perfil en más de (1-margen_ml). SEMANAL (repetir la última semana) es solo referencia.
def _tareas_backtest():
    for k in ACTIVAS:
        m,var=k;w=WINNERS[k];cut=w['last']-pd.Timedelta(days=CFG['test_days']+1)
        idx=pd.date_range(cut+pd.Timedelta(minutes=15),periods=H_SEL,freq='15min',name='Fecha')
        cols_cal=[c for c in w['columns'] if not es_columna_dinamica(c)]
        cal=features_calendario(idx,FERIADOS,CLIMATOLOGIA_TEMPERATURA,PBI_MENSUAL,POBLACION_ANUAL)
        yield dict(clave=k,ruta_modelo=w['ruta_modelo'],columnas=w['columns'],
                   historia=historia_recursion(m,var,cut).to_numpy()[-BUFFER_HISTORIA:],
                   calendario_columnas=cols_cal,calendario_valores=cal[cols_cal].to_numpy(float),indice_futuro=idx)
_ruta_bt,_ruta_sel=RUN/'tablas/backtest_recursivo.csv',RUN/'tablas/seleccion_metodo.csv'
if CFG.get('reanudar_run') and _ruta_bt.exists() and _ruta_sel.exists() and 'metodo' in pd.read_csv(_ruta_bt,nrows=1).columns:
    BT_DF=pd.read_csv(_ruta_bt);SELECCION=pd.read_csv(_ruta_sel);print('Backtest y selección de método recuperados de la corrida previa')
else:
    _ml_bt={}
    for k,pred,seg,error in proyectar_series(_tareas_backtest(),n_jobs=CFG['workers_pronostico']):
        if error: log('backtest',f'{k[0]}/{k[1]}','ERROR',error)
        else: _ml_bt[k]=pred
    print(f'Backtest ML: {len(_ml_bt)}/{len(ACTIVAS)} series')
    BT=[];SEL=[]
    for k in ACTIVAS:
        m,var=k;w=WINNERS[k];cut=w['last']-pd.Timedelta(days=CFG['test_days']+1)
        idx=pd.date_range(cut+pd.Timedelta(minutes=15),periods=H_SEL,freq='15min')
        hist_bt=historia_recursion(m,var,cut);real=load_meter(m)[var].reindex(idx).to_numpy()
        cand=_perfiles(hist_bt,idx);cand['SEMANAL']=np.resize(hist_bt.tail(672).to_numpy(),H_SEL)
        if k in _ml_bt: cand['ML']=_ml_bt[k]
        fuera=bool(idx[0]>=w['test_inicio']) if w.get('test_inicio') is not None else None
        std_real=float(np.nanstd(real));fila=dict(medidor=m,variable=var,modelo_ml=w['name'],fuera_de_muestra=fuera,std_real=std_real)
        for nombre,p in cand.items():
            for h in CFG['backtest_horizontes']:
                BT.append(dict(medidor=m,variable=var,modelo_ml=w['name'],metodo=nombre,horizonte=h,fuera_de_muestra=fuera,
                               std_real=std_real,**metrics(real[:h],p[:h])))
            e=real-p;ok=np.isfinite(e)   # banda empírica (5%-95%) del error a H_SEL pasos: exploratoria
            if ok.sum()>=96: fila[f'lo_{nombre}'],fila[f'hi_{nombre}']=np.quantile(e[ok],[.05,.95])
            fila[f'rmse_{nombre}']=metrics(real,p)['RMSE']
        perfiles={n:fila.get(f'rmse_{n}',np.nan) for n in ('PERFIL_ANCLADO','PERFIL_CALENDARIO')}
        mejor_perfil=min(perfiles,key=lambda n:(np.isnan(perfiles[n]),perfiles[n]));rml=fila.get('rmse_ML',np.nan)
        fila['mejor_perfil']=mejor_perfil
        fila['metodo']='ML' if (np.isfinite(rml) and np.isfinite(perfiles[mejor_perfil]) and rml<CFG['margen_ml']*perfiles[mejor_perfil]) else mejor_perfil
        SEL.append(fila)
    BT_DF=pd.DataFrame(BT);SELECCION=pd.DataFrame(SEL)
    BT_DF.to_csv(_ruta_bt,index=False);SELECCION.to_csv(_ruta_sel,index=False)
SEL_IDX={(r['medidor'],r['variable']):r for r in SELECCION.to_dict('records')}
print('Método elegido por serie:',SELECCION.metodo.value_counts().to_dict())

def _banda(k,metodo):
    s=SEL_IDX[k];lo,hi=s.get(f'lo_{metodo}'),s.get(f'hi_{metodo}')
    return (0.0 if pd.isna(lo) else float(lo)),(0.0 if pd.isna(hi) else float(hi))
def construir_frame(k,pred_ml=None):
    m,var=k;w=WINNERS[k];s=SEL_IDX[k];hist=historia_recursion(m,var,w['last'])
    idx=pd.date_range(w['last']+pd.Timedelta(minutes=15),FIN_PLAN,freq='15min',name='Fecha')
    perf=_perfiles(hist,idx);fuente=s['metodo'];fallback=''
    if fuente=='ML':
        san_ml=evaluar_sanidad(pd.Series(pred_ml,index=idx),hist,CFG['umbral_sanidad'])
        if san_ml['estado']=='REVISAR':   # guarda de sanidad: nunca se recorta en silencio
            fallback='ML descartado por sanidad ('+san_ml['marcas']+')';fuente=s['mejor_perfil']
    final=np.asarray(pred_ml) if fuente=='ML' else perf[fuente]
    san=evaluar_sanidad(pd.Series(final,index=idx),hist,CFG['umbral_sanidad']);lo,hi=_banda(k,fuente)
    return pd.DataFrame({'Fecha':idx,'pred':final,'banda_inf_exploratoria':final+lo,'banda_sup_exploratoria':final+hi,
        'pred_ml':np.asarray(pred_ml) if pred_ml is not None else np.nan,
        'pred_perfil_anclado':perf['PERFIL_ANCLADO'],'pred_perfil_calendario':perf['PERFIL_CALENDARIO'],
        'fuente':pd.Categorical([fuente]*len(idx)),'estado_sanidad':pd.Categorical([san['estado']]*len(idx)),
        'marcas_sanidad':pd.Categorical([san['marcas']]*len(idx)),'fallback':pd.Categorical([fallback]*len(idx)),
        'desvio_media':san['desvio_media'],'ratio_std':san['ratio_std']})
def construir_frame_constante(k):
    m,var=k;w=WINNERS[k];hist=historia_recursion(m,var,w['last'])
    idx=pd.date_range(w['last']+pd.Timedelta(minutes=15),FIN_PLAN,freq='15min',name='Fecha')
    c=np.full(len(idx),float(hist.tail(96*30).median()))
    return pd.DataFrame({'Fecha':idx,'pred':c,'banda_inf_exploratoria':c,'banda_sup_exploratoria':c,'pred_ml':np.nan,
        'pred_perfil_anclado':c,'pred_perfil_calendario':c,'fuente':pd.Categorical([ESTADO_CONSTANTE]*len(idx)),
        'estado_sanidad':pd.Categorical(['NO_APLICA']*len(idx)),'marcas_sanidad':pd.Categorical(['']*len(idx)),
        'fallback':pd.Categorical(['']*len(idx)),'desvio_media':np.nan,'ratio_std':np.nan})

def registrar_proyeccion(k,frame):
    m,var=k;w=WINNERS[k];hist=historia_recursion(m,var,w['last']);FORECASTS[k]=frame
    SANIDAD.append(dict(medidor=m,variable=var,modelo_ml=w['name'],fuente=str(frame['fuente'].iloc[0]),
        estado=str(frame['estado_sanidad'].iloc[0]),marcas=str(frame['marcas_sanidad'].iloc[0]),
        fallback=str(frame['fallback'].iloc[0]),desvio_media=frame['desvio_media'].iloc[0],ratio_std=frame['ratio_std'].iloc[0]))
    frame_por_fecha=frame.set_index('Fecha')
    # Curva de proyección mensual: a resolución de 15 min un horizonte de años es
    # ilegible: se promedia por mes para ver claramente EN QUÉ MESES crece o decrece
    # la demanda proyectada, con el histórico reciente de contexto y la banda exploratoria.
    try:
        hist_mensual=hist.resample('MS').mean().tail(24)
        pred_mensual=frame_por_fecha['pred'].resample('MS').mean()
        banda_inf_mensual=frame_por_fecha['banda_inf_exploratoria'].resample('MS').mean()
        banda_sup_mensual=frame_por_fecha['banda_sup_exploratoria'].resample('MS').mean()
        # Se guarda para la figura COMBINADA por medidor (P+Q en un solo panel, mismo
        # formato que curva_{medidor}.png), generada despues de este bucle una vez que
        # ambas variables del medidor ya tienen su pronostico mensual calculado.
        MENSUAL[k]=dict(hist=hist_mensual,pred=pred_mensual,
            banda_inf=banda_inf_mensual,banda_sup=banda_sup_mensual,last=w['last'])
        # Variación mensual (%): identifica directamente los meses donde la demanda proyectada
        # CRECE (verde) frente a los meses donde, en parte, se consume MENOS que el mes previo
        # (rojo) -- y por construcción es consistente con la curva de arriba, no otra fuente.
        crecimiento=pred_mensual.pct_change()*100
        fig,ax=plt.subplots(figsize=(14,3))
        ax.bar(crecimiento.index,crecimiento.values,width=20,
               color=['tab:green' if v>=0 else 'tab:red' for v in crecimiento.fillna(0).values])
        ax.axhline(0,color='black',lw=.8)
        ax.set_ylabel('% cambio mensual');ax.set_title(f'{m} — {var}: variación mensual proyectada (verde=crece, rojo=decrece)')
        fig.tight_layout();fig.savefig(RUN/f'figuras/crecimiento_{TOKENS[m]}_{var}.png',dpi=120);plt.close(fig)
    except Exception as exc: log('figura_proyeccion',f'{m}/{var}','ERROR',exc)
    # Un archivo por año calendario; por la convención de fin de intervalo (celda 12) el
    # timestamp 00:00 del 1-ene pertenece al año que termina.
    for year in range(w['last'].year,FIN_PLAN.year+1):
        chunk=frame_por_fecha.loc[(frame_por_fecha.index>pd.Timestamp(year,1,1))&(frame_por_fecha.index<=pd.Timestamp(year+1,1,1))]
        if chunk.empty:continue
        chunk.reset_index().to_parquet(RUN/'pronosticos'/f'operativo_{TOKENS[m]}_{var}_{year}.parquet',index=False)
        if CFG['exportar_excel']:chunk[['pred','banda_inf_exploratoria','banda_sup_exploratoria']].reset_index().to_excel(
            RUN/'pronosticos'/f'operativo_{TOKENS[m]}_{var}_{year}.xlsx',engine='xlsxwriter')

def _ruta_serie(k): return RUN/'pronosticos'/f'serie_{TOKENS[k[0]]}_{k[1]}.parquet'
def _guardar_serie(k,frame): frame.to_parquet(_ruta_serie(k),index=False)
_pendientes_ml=[]
for k in ACTIVAS+CONSTANTES:
    # Checkpoint por serie: al reanudar, lo ya proyectado no se recalcula (borre pronosticos/serie_*.parquet
    # si cambia la lógica de proyección).
    if CFG.get('reanudar_run') and _ruta_serie(k).exists(): registrar_proyeccion(k,pd.read_parquet(_ruta_serie(k)))
    elif k in CONSTANTES:
        frame=construir_frame_constante(k);_guardar_serie(k,frame);registrar_proyeccion(k,frame)
    elif SEL_IDX[k]['metodo']=='ML': _pendientes_ml.append(k)   # el ML solo se proyecta a 2027 donde ganó el backtest
    else:
        frame=construir_frame(k);_guardar_serie(k,frame);registrar_proyeccion(k,frame)
print(f'{len(FORECASTS)} series proyectadas (perfiles/checkpoints/constantes); {len(_pendientes_ml)} por proyectar con ML ({CFG["workers_pronostico"]} procesos)')

def _tareas():
    for k in _pendientes_ml:
        m,var=k;w=WINNERS[k];hist=historia_recursion(m,var,w['last'])
        idx=pd.date_range(w['last']+pd.Timedelta(minutes=15),FIN_PLAN,freq='15min',name='Fecha')
        cols_cal=[c for c in w['columns'] if not es_columna_dinamica(c)]
        yield dict(clave=k,ruta_modelo=w['ruta_modelo'],columnas=w['columns'],historia=hist.to_numpy()[-BUFFER_HISTORIA:],
                   calendario_columnas=cols_cal,calendario_valores=CAL_GLOBAL.loc[idx[0]:idx[-1],cols_cal].to_numpy(float),
                   indice_futuro=idx)
_t0=time.time();_hechas=0
for k,pred,seg,error in proyectar_series(_tareas(),n_jobs=CFG['workers_pronostico']):
    _hechas+=1
    try:
        if error:   # el ML falló para esta serie: se usa el mejor perfil y queda registrado
            log('pronostico',f'{k[0]}/{k[1]}','ADVERTENCIA','ML falló ('+error+'); se usa '+SEL_IDX[k]['mejor_perfil'])
            SEL_IDX[k]['metodo']=SEL_IDX[k]['mejor_perfil'];frame=construir_frame(k)
        else: frame=construir_frame(k,pred)
        _guardar_serie(k,frame);registrar_proyeccion(k,frame)
        print(f'[{_hechas}/{len(_pendientes_ml)}] {k[0]}/{k[1]}: {len(frame)} pasos, fuente {frame["fuente"].iloc[0]} '
              f'({WINNERS[k]["name"]}, {seg:.0f} s); transcurrido {(time.time()-_t0)/60:.1f} min',flush=True)
    except Exception as exc: log('pronostico',f'{k[0]}/{k[1]}','ERROR',exc);print(f'{k}: ERROR {exc}',flush=True)

SANIDAD_DF=pd.DataFrame(SANIDAD);SANIDAD_DF.to_csv(RUN/'tablas/sanidad_proyecciones.csv',index=False)
_estado_proy=pd.DataFrame([dict(medidor=m,variable=v,modelo_ml=w['name'],estado_serie=w['estado'],motivo=w['motivo'],
    proyectada=(m,v) in FORECASTS,fuente=str(FORECASTS[(m,v)]['fuente'].iloc[0]) if (m,v) in FORECASTS else '')
    for (m,v),w in WINNERS.items()])
_estado_proy.to_csv(RUN/'tablas/estado_proyeccion.csv',index=False)
print(f'Proyecciones: {len(FORECASTS)} series; fuente: {_estado_proy.fuente.value_counts().to_dict()}; '
      f'sanidad REVISAR: {(SANIDAD_DF.estado=="REVISAR").sum() if len(SANIDAD_DF) else 0}')

# Curva de PROYECCION combinada por medidor: mismo formato que curva_{medidor}.png
# (panel unico, P=azul/Q=verde/S=morado superpuestos) -- el tramo observado en linea
# solida continua DIRECTAMENTE en linea punteada del mismo color para el pronostico,
# con una linea vertical marcando el ultimo dato observado. No existe un modelo ML
# que pronostique S directamente (solo P y Q se entrenan); S se DERIVA igual que
# S_calculada en la limpieza historica (celda 10): S=hypot(P,Q). Para el tramo
# observado se usa S_medida real cuando el medidor la trae (mismo dato que curva_);
# para el pronostico, unicamente hypot(P_pred,Q_pred), porque no existe S medida futura.
COLOR_VARIABLE={'P':'tab:blue','Q':'tab:green','S_medida':'tab:purple'}
ETIQUETA_VARIABLE={'P':'P','Q':'Q','S_medida':'S'}
_backend_prev=matplotlib.get_backend();plt.switch_backend('Agg')
for m in MEDIDORES:
    disponibles=[var for var in ('P','Q','S_medida') if (m,var) in MENSUAL]
    if not disponibles: continue
    try:
        fig,ax=plt.subplots(figsize=(14,5))
        for var in disponibles:
            bloque=MENSUAL[(m,var)];color=COLOR_VARIABLE[var];etiqueta=ETIQUETA_VARIABLE[var]
            bloque['hist'].plot(ax=ax,color=color,lw=1.3,label=f'{etiqueta} observado (prom. mensual)')
            bloque['pred'].plot(ax=ax,color=color,lw=1.3,ls='--',label=f'{etiqueta} proyección (prom. mensual, {FORECASTS[(m,var)]["fuente"].iloc[0]})')
            ax.fill_between(bloque['pred'].index,bloque['banda_inf'],bloque['banda_sup'],color=color,alpha=.12)
        # Respaldo SOLO si el medidor no tiene KVA real entrenado (5/50): se sigue
        # derivando S=hypot(P,Q) para no perder la 3ra curva en esos casos.
        if 'S_medida' not in disponibles and 'P' in disponibles and 'Q' in disponibles:
            bp,bq=MENSUAL[(m,'P')],MENSUAL[(m,'Q')]
            d_m=load_meter(m)
            s_medida_mensual=(d_m['S_medida'].resample('MS').mean().tail(24)
                if 'S_medida' in d_m.columns and d_m['S_medida'].notna().any() else None)
            hist_S=s_medida_mensual if s_medida_mensual is not None else np.hypot(bp['hist'],bq['hist'])
            pred_S=pd.Series(np.hypot(bp['pred'],bq['pred']),index=bp['pred'].index)
            etiqueta_hist_s='S medido' if s_medida_mensual is not None else 'S calculado (hypot P,Q)'
            hist_S.plot(ax=ax,color='tab:purple',lw=1.3,label=f'{etiqueta_hist_s} (prom. mensual)')
            pred_S.plot(ax=ax,color='tab:purple',lw=1.3,ls='--',label='S proyección (derivado hypot P,Q -- sin KVA real, prom. mensual)')
        ultimo_observado=max(MENSUAL[(m,var)]['last'] for var in disponibles)
        ax.axvline(ultimo_observado,color='gray',ls=':',lw=.8)
        ax.set_ylabel('Potencia (kW / kvar / kVA)')
        ax.set_title(f'{m} — proyección mensual hasta {FIN_PLAN.date()} ({"/".join(ETIQUETA_VARIABLE[v] for v in disponibles)} continuando desde el último dato observado)')
        ax.legend(loc='upper left',fontsize=8)
        fig.tight_layout();fig.savefig(RUN/f'figuras/proyeccion_{TOKENS[m]}.png',dpi=120);plt.close(fig)
    except Exception as exc: log('figura_proyeccion_medidor',m,'ERROR',exc)
plt.close('all');plt.switch_backend(_backend_prev)


## 20b · Tasa de crecimiento (CAGR) por medidor y del sistema total

Compara el promedio del primer año histórico completo (`CFG['fecha_inicio']`) contra el promedio del último año completo del horizonte de pronóstico (`CFG['fin_plan']`), por medidor y para el sistema total (suma de todos los medidores con pronóstico). Es la tasa de crecimiento que el modelo ML **implica** en su propio pronóstico -- no un supuesto elegido a mano -- útil para comparar contra los escenarios de crecimiento definidos manualmente en la sección de planificación.

In [ ]:
# --- Tasa de crecimiento anual compuesta (CAGR): por medidor y del sistema total,
# primer año histórico completo vs. ultimo año completo del horizonte de pronostico.
# Esta es la tasa que el modelo ML YA implica en su pronostico, no un supuesto adicional.
anio_inicio_hist=pd.Timestamp(CFG['fecha_inicio']).year
anio_fin_plan=pd.Timestamp(CFG['fin_plan']).year-1  # ultimo anio completo antes de fin_plan
def _cagr(v_inicio,v_fin,n_anios):
    if pd.isna(v_inicio) or pd.isna(v_fin) or v_inicio<=0 or n_anios<=0: return np.nan
    return (v_fin/v_inicio)**(1/n_anios)-1

CRECIMIENTO=[]
for (m,var),frame in FORECASTS.items():
    hist=load_meter(m)[var]
    prom_inicio=hist[hist.index.year==anio_inicio_hist].mean()
    pron=frame.set_index('Fecha')['pred']
    prom_fin=pron[pron.index.year==anio_fin_plan].mean()
    cagr=_cagr(prom_inicio,prom_fin,anio_fin_plan-anio_inicio_hist)
    CRECIMIENTO.append(dict(medidor=m,variable=var,
        promedio_anio_inicio=prom_inicio,anio_inicio=anio_inicio_hist,
        promedio_anio_fin=prom_fin,anio_fin=anio_fin_plan,
        cagr_anual_pct=cagr*100 if pd.notna(cagr) else np.nan))

# Sistema total (P y Q por separado): suma de todos los medidores CON pronostico de esa
# variable -- ni mas ni menos que los que efectivamente se entrenaron y pronosticaron.
for var_total in ('P','Q'):
    medidores_var=[m for (m,var) in FORECASTS if var==var_total]
    if not medidores_var: continue
    series_pron=[FORECASTS[(m,var_total)].set_index('Fecha')['pred'].rename(m) for m in medidores_var]
    total_pron=pd.concat(series_pron,axis=1).sum(axis=1,min_count=1)
    series_hist=[load_meter(m)[var_total].rename(m) for m in medidores_var]
    total_hist=pd.concat(series_hist,axis=1).sum(axis=1,min_count=1)
    prom_inicio=total_hist[total_hist.index.year==anio_inicio_hist].mean()
    prom_fin=total_pron[total_pron.index.year==anio_fin_plan].mean()
    cagr=_cagr(prom_inicio,prom_fin,anio_fin_plan-anio_inicio_hist)
    CRECIMIENTO.append(dict(medidor='SISTEMA_TOTAL',variable=var_total,
        promedio_anio_inicio=prom_inicio,anio_inicio=anio_inicio_hist,
        promedio_anio_fin=prom_fin,anio_fin=anio_fin_plan,
        cagr_anual_pct=cagr*100 if pd.notna(cagr) else np.nan))

pd.DataFrame(CRECIMIENTO).to_csv(RUN/'tablas/tasa_crecimiento.csv',index=False)
_fila_sistema_p=[c for c in CRECIMIENTO if c['medidor']=='SISTEMA_TOTAL' and c['variable']=='P']
_cagr_sistema=_fila_sistema_p[0]['cagr_anual_pct'] if _fila_sistema_p else np.nan
print(f'Tasa de crecimiento calculada para {len(FORECASTS)} series + sistema total '
      f'({anio_inicio_hist}->{anio_fin_plan}). CAGR sistema total P: '
      f'{_cagr_sistema:.2f}% anual' if pd.notna(_cagr_sistema) else 'sin dato suficiente')


## 21 · Backtesting multistep

Resumen del backtest recursivo **sin reentrenar** (se calcula en la sección 20 para poder elegir el método de cada serie): últimos `test_days`+1 días del tramo de prueba, horizontes `CFG['backtest_horizontes']` (96 = 1 día, 672 = 1 semana, 2688 = 28 días), para el ML y para los perfiles. Se registra por separado del error rolling de un paso; `fuera_de_muestra` indica si la ventana quedó fuera del entrenamiento.

In [ ]:
# Resumen del backtest multi-paso (calculado en la sección 20, donde se usa para elegir el método de
# cada serie). RMSE relativo = RMSE / desvío estándar real de la ventana: 1.0 equivale a no saber nada
# más que la media; menor es mejor. Sin reentrenar: el modelo guardado nunca vio esos días
# (`fuera_de_muestra` lo indica por serie).
_bt=BT_DF.copy();_bt['rmse_rel']=_bt.RMSE/_bt.std_real.replace(0,np.nan)
print('RMSE relativo al desvío real (mediana entre series) por horizonte (pasos de 15 min) y método:')
print(_bt.pivot_table(index='horizonte',columns='metodo',values='rmse_rel',aggfunc='median').round(3).to_string())
_h=_bt[_bt.horizonte==H_SEL].pivot_table(index=['medidor','variable'],columns='metodo',values='RMSE')
print(f'\nVictorias por método a {H_SEL} pasos entre ML/PERFIL_ANCLADO/PERFIL_CALENDARIO:',
      _h[[c for c in ('ML','PERFIL_ANCLADO','PERFIL_CALENDARIO') if c in _h]].idxmin(axis=1).value_counts().to_dict())
print('Método elegido por serie (el ML solo si mejora al mejor perfil en más de',f"{1-CFG['margen_ml']:.0%}):",SELECCION.metodo.value_counts().to_dict())
if 'fuera_de_muestra' in SELECCION: print('Series con la ventana de backtest fuera del entrenamiento:',int(SELECCION.fuera_de_muestra.fillna(False).sum()),'de',len(SELECCION))


## 22 · Planificación hasta el cierre del horizonte

Perfil calendario histórico por mes/día/hora + tasas suministradas. Se mantiene separado del ML operativo; sin tasas no hay proyección larga.


In [ ]:
from prodma.sanidad import calendar_profile  # perfil calendario vectorizado (en el paquete)
def _calendar_profile_referencia(s,index):
    # Referencia lenta punto a punto (semántica original: mes exacto, luego reserva dow/bloque, luego mediana global)
    train=s.dropna().tail(366*96)
    tab=train.groupby([train.index.month,train.index.dayofweek,train.index.hour*4+train.index.minute//15]).median()
    fallback=train.groupby([train.index.dayofweek,train.index.hour*4+train.index.minute//15]).median()
    return np.array([tab.get((t.month,t.dayofweek,t.hour*4+t.minute//15),
        fallback.get((t.dayofweek,t.hour*4+t.minute//15),train.median())) for t in index])
_s_prueba=pd.Series(np.arange(400*96,dtype=float)%50+np.random.RandomState(0).randn(400*96),
    index=pd.date_range('2025-01-01',periods=400*96,freq='15min'))
for _idx_prueba in (pd.date_range('2026-02-05','2026-02-06',freq='15min')[:-1],   # fuera del mes de entrenamiento -> nivel de reserva
                    pd.date_range('2025-11-05','2025-11-06',freq='15min')[:-1]):  # dentro del historial -> nivel exacto
    assert np.allclose(calendar_profile(_s_prueba,_idx_prueba),
        _calendar_profile_referencia(_s_prueba,_idx_prueba),equal_nan=True)
print('calendar_profile vectorizado verificado contra referencia punto a punto (nivel exacto y nivel de reserva)')

PLAN_SUMMARY=[]
if not CFG['escenarios']:
    log('planificacion',str(pd.Timestamp(CFG['fin_plan']).year-1),'PENDIENTE','Configurar tasas sustentadas en CFG[escenarios]')
else:
  for m in TARGETS:
    if (m,'P') not in WINNERS or (m,'Q') not in WINNERS:continue
    hist=load_meter(m);last=min(WINNERS[(m,'P')]['last'],WINNERS[(m,'Q')]['last'])
    for scenario,growth in CFG['escenarios'].items():
      assert np.isfinite(growth) and growth>-1
      for year in range(last.year,pd.Timestamp(CFG['fin_plan']).year+1):
        begin=max(last+pd.Timedelta(minutes=15),pd.Timestamp(year,1,1,0,15))
        end=min(pd.Timestamp(year+1,1,1),pd.Timestamp(CFG['fin_plan']))
        if begin>end:continue
        idx=pd.date_range(begin,end,freq='15min',name='Fecha')
        factor=(1+growth)**((idx-last).total_seconds()/(365.25*86400))
        frame=pd.DataFrame(index=idx)
        for var in ('P','Q'):frame[var+'_base']=calendar_profile(hist[var].loc[:last],idx)*factor
        frame['P_nuevos']=0.;frame['Q_nuevos']=0.;pending=[]
        if CFG['incorporar_clientes']:
          if CFG['calendario_etapas'] not in ('incremental','acumulado'):raise ValueError('Confirmar significado de columnas anuales')
          additions,pending=add_new_customers(idx,m,CLIENTS)
          frame[['P_nuevos','Q_nuevos']]=additions
          for issue in pending:log('alta',issue['id'],'PENDIENTE',issue['motivo'])
        else:pending=[c['id'] for c in CLIENTS if c['medidor']==m]
        for var in ('P','Q'):frame[var+'_final']=frame[var+'_base']+frame[var+'_nuevos']
        frame['S_final']=np.hypot(frame.P_final,frame.Q_final)
        frame['FP']=frame.P_final/frame.S_final.replace(0,np.nan)
        frame['clientes_pendientes']=len(pending)
        frame['clientes_archivo_disponible']=CLIENTES_DISPONIBLE
        frame['historico_incompleto']=bool(HISTORICOS_FALTANTES)
        # PARCIAL si faltan altas por confirmar, o si el archivo de clientes no estaba disponible,
        # o si algún histórico anual requerido está ausente: ninguno de estos casos es un escenario cerrado.
        frame['estado']='PARCIAL' if (pending or not CLIENTES_DISPONIBLE or HISTORICOS_FALTANTES) else 'ESCENARIO_CONDICIONADO'
        frame.to_parquet(RUN/'pronosticos'/f'plan_{TOKENS[m]}_{key(scenario)}_{year}.parquet')
        if CFG['exportar_excel']:frame.to_excel(RUN/'pronosticos'/f'plan_{TOKENS[m]}_{key(scenario)}_{year}.xlsx',engine='xlsxwriter')
        PLAN_SUMMARY.append(dict(medidor=m,escenario=scenario,anio=year,max_P=frame.P_final.max(),
             max_S=frame.S_final.max(),fecha_max=str(frame.P_final.idxmax()),
             energia_perfil_kWh=float(frame.P_final.sum()*.25),clientes_pendientes=len(pending),
             clientes_archivo_disponible=CLIENTES_DISPONIBLE,historico_incompleto=bool(HISTORICOS_FALTANTES),
             estado=frame['estado'].iloc[0]))
pd.DataFrame(PLAN_SUMMARY).to_csv(RUN/'tablas/resumen_planificacion.csv',index=False)


## 23 · Exportación operacional conjunta

Integra P y Q con los nuevos clientes (definidos por kW). S_medida NO se integra aquí -- los clientes nuevos no tienen un concepto de "S_nuevos" propio en los datos disponibles -- pero su pronóstico propio (cuando el medidor tiene KVA real) sigue disponible en `pronosticos/operativo_*_S_medida_*`. Una matriz por conjunto de fechas, conservando huecos cuando los últimos registros difieren. Se grafica además la curva consolidada de la demanda TOTAL mensual (base vs. base+nuevos clientes) y su variación mes a mes, para que una caída parcial en algún medidor no quede oculta en el total de la red.


In [ ]:
if FORECASTS:
    final_columns={};base_columns={};CLIENT_STATUS=[]
    for (m,var),d in FORECASTS.items():
        if var=='S_medida':
            continue  # nuevos clientes se definen por kW; S no tiene 'S_nuevos' propio
        base=d.set_index('Fecha')['pred'];idx=base.index
        if not CFG['incorporar_clientes']:
            additions=pd.DataFrame({'P_nuevos':0.,'Q_nuevos':0.},index=idx)
            pending=[{'id':c['id'],'motivo':'integración desactivada'} for c in CLIENTS if c['medidor']==m]
        elif not CLIENTES_DISPONIBLE:
            additions=pd.DataFrame({'P_nuevos':0.,'Q_nuevos':0.},index=idx)
            pending=[{'id':'(archivo_nuevosclientes)','motivo':'archivo de nuevos clientes no disponible en la carpeta local'}]
        else:
            additions,pending=add_new_customers(idx,m,CLIENTS)
        total=base+additions[var+'_nuevos']
        # La igualdad total-base==incremento es tautológica por construcción; lo que se verifica
        # realmente es que la suma no introduce NaN donde la base y el incremento eran válidos.
        assert not total.loc[base.notna()&additions[var+'_nuevos'].notna()].isna().any(),'NaN inesperado al integrar nuevos clientes'
        detail=pd.DataFrame({var+'_base':base,var+'_nuevos':additions[var+'_nuevos'],var+'_final':total})
        detail['clientes_pendientes']=len(pending)
        detail['clientes_archivo_disponible']=CLIENTES_DISPONIBLE
        detail.to_parquet(RUN/'pronosticos'/f'total_{TOKENS[m]}_{var}.parquet')
        final_columns[f'{TOKENS[m]}_{var}']=total;base_columns[f'{TOKENS[m]}_{var}']=base
        for issue in pending:CLIENT_STATUS.append(dict(medidor=m,variable=var,**issue))
    pd.DataFrame(CLIENT_STATUS).to_csv(RUN/'tablas/clientes_no_integrados.csv',index=False)
    pd.concat(base_columns,axis=1).to_parquet(RUN/'pronosticos/Matriz_Operativa_Base.parquet')
    wide=pd.concat(final_columns,axis=1)
    wide.index.name='Fecha';wide.to_parquet(RUN/'pronosticos/Matriz_Operativa.parquet')
    if CFG['exportar_excel']:wide.to_excel(RUN/'pronosticos/Matriz_Operativa.xlsx',engine='xlsxwriter')
    print('Perfil TOTAL = base + incrementos constantes por Medidor:',wide.shape)
    print('Etapas pendientes de fecha/confirmación:',len(CLIENT_STATUS))
    if not CLIENTES_DISPONIBLE:
        print('Nota: archivo de nuevos clientes no disponible; TOTAL == BASE en esta ejecución.')
    # Curva de proyección CONSOLIDADA (todos los medidores + nuevos clientes, promedio
    # mensual): muestra en qué meses/años crece la demanda TOTAL de la red, y separa
    # cuánto viene de la base (puede crecer o decrecer medidor a medidor) frente a los
    # nuevos clientes, para que una caída parcial en la base no quede oculta en el total.
    try:
        p_cols_total=[c for c in wide.columns if c.endswith('_P')]
        p_cols_base=[c for c in pd.concat(base_columns,axis=1).columns if c.endswith('_P')]
        if p_cols_total:
            base_wide=pd.concat(base_columns,axis=1)
            total_mensual=wide[p_cols_total].sum(axis=1,min_count=1).resample('MS').mean()
            base_mensual=base_wide[p_cols_base].sum(axis=1,min_count=1).resample('MS').mean()
            nuevos_mensual=(total_mensual-base_mensual).clip(lower=0)
            fig,ax=plt.subplots(figsize=(14,5))
            base_mensual.plot(ax=ax,color='tab:blue',label='base (perfil histórico, sin nuevos clientes)')
            total_mensual.plot(ax=ax,color='darkorange',label='total (base + nuevos clientes)')
            ax.legend();ax.set_ylabel('P (kW)');ax.set_title('Proyección consolidada: demanda total mensual, todos los medidores')
            fig.tight_layout();fig.savefig(RUN/'figuras/proyeccion_consolidada_mensual.png',dpi=140);plt.close(fig)
            crecimiento_total=total_mensual.pct_change()*100
            fig,ax=plt.subplots(figsize=(14,3))
            ax.bar(crecimiento_total.index,crecimiento_total.values,width=20,
                   color=['tab:green' if v>=0 else 'tab:red' for v in crecimiento_total.fillna(0).values])
            ax.axhline(0,color='black',lw=.8)
            ax.set_ylabel('% cambio mensual');ax.set_title('Variación mensual de la demanda TOTAL consolidada (verde=crece, rojo=decrece)')
            fig.tight_layout();fig.savefig(RUN/'figuras/crecimiento_consolidado_mensual.png',dpi=140);plt.close(fig)
    except Exception as exc: log('figura_proyeccion_consolidada','TOTAL','ERROR',exc)

## 24 · Verificación final

Verifica archivos producidos. No declara entrenamientos ni altas pendientes como completos.


In [ ]:
checks=[]
for p in (RUN/'pronosticos').glob('*.parquet'):
    d=pd.read_parquet(p);idx=d['Fecha'] if 'Fecha' in d else d.index
    checks.append(dict(archivo=p.name,filas=len(d),fechas_unicas=bool(pd.Index(idx).is_unique)))
pd.DataFrame(checks).to_csv(RUN/'tablas/verificacion_salidas.csv',index=False)
# Verificación de CADA proyección operativa: cubre exactamente hasta el cierre del horizonte, sin huecos,
# finita y no plana (salvo las series constantes declaradas).
VERIF=[]
for (m,var),frame in FORECASTS.items():
    f=frame.set_index('Fecha')['pred'];fuente=str(frame['fuente'].iloc[0])
    VERIF.append(dict(medidor=m,variable=var,fuente=fuente,pasos=len(f),inicio=f.index.min(),fin=f.index.max(),
        termina_en_fin_plan=bool(f.index.max()==FIN_PLAN),indice_unico=bool(f.index.is_unique),
        paso_15min=bool((f.index.to_series().diff().dropna()==pd.Timedelta(minutes=15)).all()),
        finitos=bool(np.isfinite(f.to_numpy()).all()),no_plana=bool(f.std()>1e-9) or fuente==ESTADO_CONSTANTE,
        min_kW=float(f.min()),max_kW=float(f.max())))
VERIF=pd.DataFrame(VERIF);VERIF.to_csv(RUN/'tablas/verificacion_proyecciones.csv',index=False)
_criterios=['termina_en_fin_plan','indice_unico','paso_15min','finitos','no_plana']
_incumplen=VERIF[~VERIF[_criterios].all(axis=1)] if len(VERIF) else VERIF
if len(_incumplen): log('verificacion','proyecciones','ERROR',f'{len(_incumplen)} series incumplen: '+', '.join(_incumplen.medidor+'/'+_incumplen.variable))
print(f'Proyecciones verificadas: {len(VERIF)-len(_incumplen)}/{len(VERIF)} cumplen todos los criterios')
limitaciones=['Intervalos operativos exploratorios de un paso; no calibrados multihorizonte',
    'Planificación por perfiles condicionada a tasas, fechas y altas confirmadas',
    'Clasificación barras/alimentadores debe revisarse antes de sumar la red',
    'No se ejecuta flujo de potencia: archivos son insumos, no validación de capacidad']
if HISTORICOS_FALTANTES:
    limitaciones.append('Históricos anuales ausentes en la carpeta local: '+str(HISTORICOS_FALTANTES)+'; diagnóstico PARCIAL')
if not CLIENTES_DISPONIBLE:
    limitaciones.append('Archivo de nuevos clientes ausente en la carpeta local; ninguna alta fue incorporada en esta ejecución')
limitaciones+= ['Proyección hasta el cierre de '+str(FIN_PLAN.year-1)+' con los modelos guardados (80% cronológico del histórico, sin reentrenar con el 100%). '
    'El pronóstico recursivo ML pierde el nivel en pocos días: el método de cada serie se eligió por backtest a 28 días (ML, perfil anclado o perfil calendario); '
    'no hay validación a 16 meses y ninguna proyección incluye crecimiento más allá del nivel reciente',
    f'{len(SIN_DATOS)} series sin datos recientes NO se proyectan y {len(CONSTANTES)} series constantes se proyectan como constante (ver estado_proyeccion.csv)',
    f'Método de proyección por serie (ver seleccion_metodo.csv y sanidad_proyecciones.csv): {SANIDAD_DF.fuente.value_counts().to_dict() if len(SANIDAD_DF) else {}}']
# Cierre de los criterios mínimos de aceptación estadística (secciones 15 y 16): nunca se
# ocultan aquí tampoco, igual que el resto de "limitaciones" de esta celda.
try:
    comp=pd.read_csv(RUN/'tablas/comparacion_completa.csv')
    idx_comp=comp.set_index(['medidor','variable','modelo'])
    filas_ganadoras=[idx_comp.loc[(m,var,w['name'])] for (m,var),w in WINNERS.items() if (m,var,w['name']) in idx_comp.index]
    if filas_ganadoras:
        cumplen=[bool(f['cumple_minimos']) for f in filas_ganadoras if pd.notna(f['cumple_minimos'])]
        n_cumplen=sum(cumplen);n_evaluados=len(cumplen)
        criterios_modelos_ganadores=f'{n_cumplen}/{n_evaluados} evaluados (de {len(filas_ganadoras)} series)'
        if n_evaluados and n_cumplen<n_evaluados:
            limitaciones.append('Modelos ganadores que no cumplen todos los criterios mínimos '
                '(Durbin-Watson, no autocorrelación, homocedasticidad, R2>=CFG[r2_minimo_aceptable]): '
                'ver comparacion_completa.csv, columna cumple_minimos')
    else:
        criterios_modelos_ganadores='N/D (sin series ganadoras en comparacion_completa.csv)'
except Exception as exc:
    criterios_modelos_ganadores='N/D (error al leer comparacion_completa.csv)'
    log('resumen_criterios','comparacion_completa','ERROR',exc)
try:
    ols_ref=pd.read_csv(RUN/'tablas/OLS_referencia.csv')
    criterios_ols_referencia=f"{int(ols_ref['cumple_minimos_ols'].sum())}/{len(ols_ref)}" if 'cumple_minimos_ols' in ols_ref and len(ols_ref) else 'N/D'
except Exception:
    criterios_ols_referencia='N/D (sin OLS_referencia.csv)'
summary=dict(modo=CFG['modo'],
    estado_ejecucion='PARCIAL' if (HISTORICOS_FALTANTES or not CLIENTES_DISPONIBLE) else 'COMPLETA_SEGUN_INSUMOS_DISPONIBLES',
    archivos_disponibles=len(paths),archivos_faltantes=FALTANTES,historicos_faltantes=HISTORICOS_FALTANTES,
    clientes_archivo_disponible=CLIENTES_DISPONIBLE,hojas_validas=len(INDEX),hojas_con_error=len(ISSUES),
    medidores_historicos=len(MEDIDORES),medidores_modelados=len(TARGETS),series_seleccionadas=len(WINNERS),
    pronosticos_operativos=len(FORECASTS),escenarios_configurados=list(CFG['escenarios']),
    clientes_confirmados=sum(c['confirmado'] for c in CLIENTS),etapas_clientes=len(CLIENTS),
    criterios_estadisticos_modelos_ganadores=criterios_modelos_ganadores,
    criterios_estadisticos_ols_referencia=criterios_ols_referencia,
    reanudacion_desde=CFG.get('reanudar_run'),horizonte_fin=str(FIN_PLAN),
    reentrenamientos_ml=0 if CFG.get('reanudar_run') else 'entrenados en la sección 14 de esta corrida',
    series_por_estado={e:int(n) for e,n in pd.Series([w['estado'] for w in WINNERS.values()]).value_counts().items()},
    proyecciones_verificadas=f'{len(VERIF)-len(_incumplen)}/{len(VERIF)}',
    metodo_por_serie={e:int(n) for e,n in SANIDAD_DF.fuente.value_counts().items()} if len(SANIDAD_DF) else {},
    resultados=str(RUN),limitaciones=limitaciones)
(RUN/'RESUMEN.json').write_text(json.dumps(summary,indent=2,ensure_ascii=False),encoding='utf-8')
display(pd.Series(summary));display(pd.DataFrame(LOG).tail(20))
if summary['estado_ejecucion']=='PARCIAL':
    print('ADVERTENCIA: ejecución PARCIAL por insumos ausentes; ver "limitaciones" y "archivos_faltantes" en RESUMEN.json.')
print('Finalizó la ejecución; consulte errores y pendientes. Resultados:',RUN)


## Alcance de la validación y referencias

Validación local: diez pruebas automatizadas sobre sintaxis, cuatro hojas reales de demanda2023.xlsx, fechas, valores cero, causalidad, ausencia previa de medición, incorporación de clientes, recursión estacional y particiones temporales. Prueba adicional con Ridge y Random Forest: entrenamiento CV y 96 pasos recursivos con valores finitos en una serie sintética.

**No ejecutado aquí:** entrenamiento integral de seis modelos con los cuatro históricos, ni validación eléctrica mediante flujo de potencia. XGBoost, LightGBM, CatBoost y MLP requieren la ejecución real de las celdas de entrenamiento.

**Alcance operativo de esta versión:** el pronóstico recursivo (celda 20) usa el modelo ML ganador de la celda 14, entrenado con el 80% (`train_frac`) más antiguo del histórico y evaluado honestamente con el 20% final, luego reentrenado con el 100% del histórico y proyectado cada 15 min, sin acortar el horizonte, desde el último dato observado hasta `CFG['fin_plan']` (resto de este año, 2027 y año por año hasta cumplir el horizonte). El escenario de planificación por perfiles y tasas (celda 22) es una herramienta aparte, opcional (`CFG['escenarios']`), que no reemplaza ni valida el pronóstico ML. Las bandas operativas son exploratorias (residuos del tramo de evaluación de un paso), no intervalos multihorizonte calibrados; el error de la recursión paso a paso no está acotado empíricamente para horizontes de varios años. La imputación modelística usa exclusivamente pasado (relleno corto y semana previa); no hay spline automático. Los archivos cacheados evitan releer las hojas; los modelos se guardan, pero una nueva ejecución vuelve a entrenarlos.

Las transferencias o retiros se representan mediante etapas firmadas, confirmadas y perfiles, registrando una etapa negativa en origen y positiva en destino. No se realiza clasificación topológica automática.

Referencias de implementación: [pandas read_excel](https://pandas.pydata.org/docs/reference/api/pandas.read_excel.html), [validación temporal scikit-learn](https://scikit-learn.org/stable/modules/cross_validation.html#time-series-split), [pathlib](https://docs.python.org/3/library/pathlib.html).

### Sustento académico de la actualización local

- Brown, C. (2021), tesis de maestría, *Reproducible notebooks*: documenta que declarar el entorno y controlar rutas/dependencias son condiciones centrales para reproducir notebooks (DOI: [10.62791/20195](https://doi.org/10.62791/20195)).
- Pimentel et al. (2021), artículo indexado, *Understanding and improving the quality and reproducibility of Jupyter notebooks*: sustenta limpiar estado/salidas heredadas y explicitar dependencias ([DOI 10.1007/s10664-021-09961-9](https://doi.org/10.1007/s10664-021-09961-9)).
- Cerqueira, Torgo y Mozetič (2020), artículo científico, *Evaluating time series forecasting models*: respalda evaluación fuera de muestra que preserve el orden temporal en series no estacionarias ([arXiv 1905.11744](https://arxiv.org/abs/1905.11744)).
- Hong, T. (2010), tesis doctoral, *Short Term Electric Load Forecasting*: fundamento del tratamiento de pronóstico de carga eléctrica y comparación de modelos ([North Carolina State University](https://repository.lib.ncsu.edu/items/c8f6d98e-3422-4a0b-9d5e-91cfeaa0de87)).
- Salehimehr et al. (2022), revisión indexada sobre pronóstico de carga de corto plazo con IA ([DOI 10.1049/tje2.12183](https://doi.org/10.1049/tje2.12183)).

## 25 · Tabla de demandas por medidor para escenarios de operación DIgSILENT PowerFactory

Máxima, promedio y mínima demanda por medidor en un **año base común** (no un año distinto por medidor, para que el escenario represente el mismo periodo en toda la red). Se calculan solo con `P`/`Q` **observados** (no con la serie imputada de la celda 10), para no fabricar un punto de operación con datos que no fueron medidos. El año base se detecta automáticamente como el más reciente en el que al menos la mitad de los medidores alcanza cobertura suficiente; puede fijarse manualmente con `ANIO_BASE`. La tabla queda en kW/kvar/kVA y también en MW/Mvar/MVA (unidades típicas de PowerFactory a nivel de red), con FP y la fecha exacta de máxima y mínima demanda para trazabilidad, y se exporta en forma larga y ancha (una fila por medidor) lista para pegar en un escenario de operación. El nombre de PowerFactory de cada elemento debe confirmarse contra `catalogo_medidores.csv`; el código de medidor no se asume idéntico al nombre del elemento en el modelo.


In [ ]:
import calendar

# Fijar aquí un año (ej. 2024) para forzar el año base; None = detección automática.
ANIO_BASE = None
UMBRAL_COBERTURA_ANIO_BASE = 0.90

def intervalos_esperados(anio):
    return 96*(366 if calendar.isleap(anio) else 365)

def cobertura_por_anio(serie_P):
    p = serie_P.dropna()
    conteo = p.groupby(p.index.year).size()
    return {int(anio): n/intervalos_esperados(int(anio)) for anio,n in conteo.items()}

# Prueba de las funciones de cobertura/año base sobre una serie sintética de un año no bisiesto completo,
# antes de aplicarlas a los medidores reales. Por la convención de fin de intervalo (cell-012), el último
# intervalo del año (23:45->00:00) queda con timestamp del 1-ene del año siguiente; por eso la cobertura
# de un año 'completo' es 35039/35040, no exactamente 1.0 -- se tolera esa única muestra de borde.
_p_prueba=pd.Series(1.0,index=pd.date_range('2023-01-01 00:15','2024-01-01 00:00',freq='15min'))
assert intervalos_esperados(2023)==35040 and intervalos_esperados(2024)==35136
_cov_prueba=cobertura_por_anio(_p_prueba)
assert set(_cov_prueba)=={2023,2024} and abs(_cov_prueba[2023]-1.0)<1e-3 and _cov_prueba[2024]<1e-3
print('Funciones de cobertura y año base verificadas (año no bisiesto completo, con el borde de fin de intervalo tolerado)')

COBERTURA_MEDIDORES={m:cobertura_por_anio(load_meter(m)['P']) for m in MEDIDORES}

if ANIO_BASE is None:
    anios_disponibles=sorted({a for cov in COBERTURA_MEDIDORES.values() for a in cov},reverse=True)
    ANIO_BASE=None
    for a in anios_disponibles:
        n_ok=sum(1 for cov in COBERTURA_MEDIDORES.values() if cov.get(a,0)>=UMBRAL_COBERTURA_ANIO_BASE)
        if n_ok>=max(1,len(MEDIDORES)//2):
            ANIO_BASE=a;break
assert ANIO_BASE is not None,'No se pudo determinar un año base con cobertura suficiente; fije ANIO_BASE manualmente'
n_ok=sum(1 for cov in COBERTURA_MEDIDORES.values() if cov.get(ANIO_BASE,0)>=UMBRAL_COBERTURA_ANIO_BASE)
print(f'Año base para escenarios DIgSILENT: {ANIO_BASE} (medidores con cobertura>={UMBRAL_COBERTURA_ANIO_BASE:.0%}: {n_ok}/{len(MEDIDORES)})')

def punto_operacion(m,anio,P,Q,ts,etiqueta,cobertura):
    p_val=float(P.loc[ts]);q_val=float(Q.loc[ts]) if pd.notna(Q.get(ts,np.nan)) else np.nan
    s_val=float(np.hypot(p_val,q_val)) if pd.notna(q_val) else np.nan
    fp_val=float(p_val/s_val) if s_val else np.nan
    return dict(medidor=m,codigo=TOKENS[m],anio_base=anio,escenario=etiqueta,fecha=str(ts),
        P_kW=p_val,Q_kvar=q_val,S_kVA=s_val,FP=fp_val,
        P_MW=p_val/1000,Q_Mvar=q_val/1000 if pd.notna(q_val) else np.nan,
        S_MVA=s_val/1000 if pd.notna(s_val) else np.nan,
        cobertura_anio_base=round(cobertura,4),cobertura_suficiente=bool(cobertura>=UMBRAL_COBERTURA_ANIO_BASE))

DEMANDAS_DIGSILENT=[]
for m in MEDIDORES:
    d=load_meter(m)
    if ANIO_BASE not in d.index.year.unique():
        log('digsilent',m,'OMITIDO',f'Sin datos en el año base {ANIO_BASE}');continue
    sub=d.loc[str(ANIO_BASE)]
    P=sub['P'].dropna();Q=sub['Q']
    if P.empty:
        log('digsilent',m,'OMITIDO',f'Sin P observado en {ANIO_BASE}');continue
    cobertura=len(P)/intervalos_esperados(ANIO_BASE)
    if cobertura<UMBRAL_COBERTURA_ANIO_BASE:
        log('digsilent',m,'ADVERTENCIA',f'Cobertura {cobertura:.1%} en {ANIO_BASE}; estadísticos con datos observados incompletos')
    ts_max=P.idxmax();ts_min=P.idxmin()
    DEMANDAS_DIGSILENT.append(punto_operacion(m,ANIO_BASE,P,Q,ts_max,'Maxima_Demanda',cobertura))
    DEMANDAS_DIGSILENT.append(punto_operacion(m,ANIO_BASE,P,Q,ts_min,'Minima_Demanda',cobertura))
    # Promedio: no corresponde a un instante real observado; P y Q se promedian cada uno por separado
    # (la Q promedio NO es la Q del instante de P promedio, a diferencia de máxima/mínima).
    p_avg=float(P.mean());q_obs=Q.dropna();q_avg=float(q_obs.mean()) if len(q_obs) else np.nan
    s_avg=float(np.hypot(p_avg,q_avg)) if pd.notna(q_avg) else np.nan
    fp_avg=float(p_avg/s_avg) if s_avg else np.nan
    DEMANDAS_DIGSILENT.append(dict(medidor=m,codigo=TOKENS[m],anio_base=ANIO_BASE,escenario='Demanda_Promedio',fecha=None,
        P_kW=p_avg,Q_kvar=q_avg,S_kVA=s_avg,FP=fp_avg,
        P_MW=p_avg/1000,Q_Mvar=q_avg/1000 if pd.notna(q_avg) else np.nan,
        S_MVA=s_avg/1000 if pd.notna(s_avg) else np.nan,
        cobertura_anio_base=round(cobertura,4),cobertura_suficiente=bool(cobertura>=UMBRAL_COBERTURA_ANIO_BASE)))

tabla_digsilent=pd.DataFrame(DEMANDAS_DIGSILENT)
assert not tabla_digsilent.empty,'No se generó ninguna fila; revisar disponibilidad de datos por medidor en el año base'
tabla_digsilent.to_csv(RUN/'tablas/Escenarios_DIgSILENT_MaxPromMin.csv',index=False)
if CFG['exportar_excel']:
    tabla_digsilent.to_excel(RUN/'tablas/Escenarios_DIgSILENT_MaxPromMin.xlsx',sheet_name='Escenarios_PowerFactory',engine='xlsxwriter')

# Vista ancha: una fila por medidor/código, columnas por escenario — lista para pegar en un escenario
# de operación de PowerFactory. Confirmar el nombre de elemento contra catalogo_medidores.csv.
tabla_digsilent_ancha=tabla_digsilent.pivot_table(index=['medidor','codigo','anio_base'],columns='escenario',
    values=['P_kW','Q_kvar','S_kVA','FP','P_MW','Q_Mvar','S_MVA','cobertura_suficiente'],aggfunc='first')
tabla_digsilent_ancha.columns=['_'.join(c) for c in tabla_digsilent_ancha.columns]
tabla_digsilent_ancha=tabla_digsilent_ancha.reset_index()
tabla_digsilent_ancha.to_csv(RUN/'tablas/Escenarios_DIgSILENT_MaxPromMin_ancha.csv',index=False)
if CFG['exportar_excel']:
    tabla_digsilent_ancha.to_excel(RUN/'tablas/Escenarios_DIgSILENT_MaxPromMin_ancha.xlsx',sheet_name='PowerFactory_ancha',engine='xlsxwriter')

display(tabla_digsilent_ancha)
print('Tabla de demandas (máxima/promedio/mínima) por medidor, año base',ANIO_BASE,'lista en',RUN/'tablas')


In [ ]:
# --- 25b · Demandas proyectadas (máx/prom/mín) por medidor para el cierre del horizonte ---
# Mismo formato que la tabla de la sección 25, pero sobre la Matriz_Operativa proyectada (P y Q
# ya incluyen las altas de clientes confirmadas), para el último año completo del horizonte.
ANIO_PROY=(FIN_PLAN-pd.Timedelta(minutes=15)).year
_ini,_fin=pd.Timestamp(ANIO_PROY,1,1),pd.Timestamp(ANIO_PROY+1,1,1)   # fin de intervalo: 00:00 del 1-ene cierra el año
DEMANDAS_PROYECTADAS=[]
for m in MEDIDORES:
    _pc,_qc=f'{TOKENS[m]}_P',f'{TOKENS[m]}_Q'
    if _pc not in wide.columns or _qc not in wide.columns: continue
    _sub=wide.loc[(wide.index>_ini)&(wide.index<=_fin),[_pc,_qc]].dropna()
    if _sub.empty: continue
    _cobertura=len(_sub)/intervalos_esperados(ANIO_PROY);P,Q=_sub[_pc],_sub[_qc]
    _fuente=str(FORECASTS[(m,'P')]['fuente'].iloc[0]) if (m,'P') in FORECASTS else ''
    for _etq,_ts in (('Maxima_Demanda',P.idxmax()),('Minima_Demanda',P.idxmin())):
        _fila=punto_operacion(m,ANIO_PROY,P,Q,_ts,_etq,_cobertura);_fila['fuente_proyeccion']=_fuente;DEMANDAS_PROYECTADAS.append(_fila)
    _p,_q=float(P.mean()),float(Q.mean());_s=float(np.hypot(_p,_q))
    DEMANDAS_PROYECTADAS.append(dict(medidor=m,codigo=TOKENS[m],anio_base=ANIO_PROY,escenario='Demanda_Promedio',fecha=None,
        P_kW=_p,Q_kvar=_q,S_kVA=_s,FP=_p/_s if _s else np.nan,P_MW=_p/1000,Q_Mvar=_q/1000,S_MVA=_s/1000,
        cobertura_anio_base=round(_cobertura,4),cobertura_suficiente=bool(_cobertura>=UMBRAL_COBERTURA_ANIO_BASE),fuente_proyeccion=_fuente))
tabla_digsilent_proyectada=pd.DataFrame(DEMANDAS_PROYECTADAS)
if len(tabla_digsilent_proyectada):
    tabla_digsilent_proyectada.to_csv(RUN/f'tablas/Escenarios_DIgSILENT_{ANIO_PROY}_proyectado.csv',index=False)
    if CFG['exportar_excel']:
        tabla_digsilent_proyectada.to_excel(RUN/f'tablas/Escenarios_DIgSILENT_{ANIO_PROY}_proyectado.xlsx',
            sheet_name='PowerFactory_proyectado',engine='xlsxwriter')
    print(f'Tabla de demandas proyectadas {ANIO_PROY}: {tabla_digsilent_proyectada.medidor.nunique()} medidores en',RUN/'tablas')
else: log('digsilent_proyectado','tabla','ADVERTENCIA','Sin filas: la matriz operativa no cubre el año final del horizonte')


## 26 · Modelos adicionales exigidos por el plan

Compara SeasonalNaive, ARIMA, SARIMA, Holt‑Winters, Prophet y LSTM, cada uno con la resolución de dataset que necesita para rendir a su potencial -- ya no se fuerza todo a mensual. Los modelos estadísticos clásicos y Prophet usan resolución **diaria** (estacionalidad semanal, período 7: informativa y a la vez tratable para SARIMA/Holt-Winters, algo que el período 672 de 15 minutos no permite). LSTM usa resolución **horaria** (período 24): se beneficia de muchas más observaciones que las ~40 que daría un remuestreo mensual. El dataset de origen es el mismo ya limpiado en la sección 10 (`load_meter`); solo cambia el remuestreo por modelo. Cada modelo pasa por la misma batería de métricas y diagnósticos estadísticos (RMSE/MAE/R², Durbin-Watson, Ljung-Box, Breusch-Pagan/White sobre predicho-vs-residuo) usada para los 6 modelos ML operativos (sección 15). SARIMAX se agrega SOLO cuando PBI (sección 12) está disponible, usando PBI como única exogena (es el único modelo de esta comparación diseñado para aceptarla); Prophet/LSTM/ARIMA/SARIMA/Holt-Winters siguen univariados por diseño de esta comparación de referencia. Los modelos de esta sección no se mezclan con el ranking operacional de 15 minutos.

In [ ]:
import sys
SRC_LOCAL=RAIZ_PROYECTO/'src'
if str(SRC_LOCAL) not in sys.path: sys.path.insert(0,str(SRC_LOCAL))
from prodma.forecast_models import create_forecaster
from prodma.variables_externas import pbi_en_fecha
from statsmodels.stats.diagnostic import acorr_ljungbox, het_breuschpagan, het_white
from statsmodels.stats.stattools import durbin_watson
import statsmodels.api as sm

# Cada familia de modelo necesita una resolucion de dataset distinta para rendir a su
# potencial: forzar TODO a mensual (version anterior) le daba a SARIMA/Holt-Winters un
# ciclo estacional (12) que no refleja el patron semanal real de la carga, y a
# Prophet/LSTM apenas ~40-45 puntos para entrenar. No se cambia el dataset FUENTE
# (load_meter ya viene limpio de la seccion 10, sin eliminar ningun dato): solo el
# remuestreo. horizonte_dias fija un horizonte de prueba comparable (~90 dias) para
# todos, expresado en la cantidad de pasos que corresponda a cada resolucion.
HORIZONTE_DIAS=90
RESOLUCION_MODELO={
    'SeasonalNaive':dict(freq='D',seasonal_periods=7,pasos_por_dia=1),
    'ARIMA':        dict(freq='D',seasonal_periods=7,pasos_por_dia=1),
    'SARIMA':       dict(freq='D',seasonal_periods=7,pasos_por_dia=1),
    'HoltWinters':  dict(freq='D',seasonal_periods=7,pasos_por_dia=1),
    'Prophet':      dict(freq='D',seasonal_periods=7,pasos_por_dia=1),
    'LSTM':         dict(freq='h',seasonal_periods=24,pasos_por_dia=24),
}
# SARIMAX solo se agrega si PBI (variable externa, seccion 12) esta disponible: es el
# unico modelo de esta comparacion disenado para aceptar exogenas -- sin PBI cargado no
# hay nada valido que pasarle, y se omite explicitamente (nunca se inventa un valor).
if PBI_MENSUAL is not None:
    RESOLUCION_MODELO['SARIMAX']=dict(freq='D',seasonal_periods=7,pasos_por_dia=1)

def _diagnosticos_residuales(real,pred,alfa):
    # Misma bateria model-agnostica que la seccion 15 (celda de diagnosticos): DW y
    # Ljung-Box sobre el residuo, Breusch-Pagan/White sobre el NIVEL PREDICHO (valido
    # para cualquier familia de modelo, no solo para OLS lineal).
    e=(real-pred).dropna()
    row={'DW':np.nan,'LB_min_p':np.nan,'BP_pred_p':np.nan,'White_pred_p':np.nan}
    try:
        if len(e)>=8:
            row['DW']=durbin_watson(e)
        lags=[v for v in (1,2,4,7,12,24) if v<len(e)//5]
        if lags:
            lb=acorr_ljungbox(e,lags=lags,return_df=True)
            row['LB_min_p']=float(lb.lb_pvalue.min())
            row['alerta_autocorrelacion']=bool(row['LB_min_p']<alfa)
        pred_alineado=pred.reindex(e.index)
        ok=pred_alineado.notna()&np.isfinite(pred_alineado)&np.isfinite(e)
        if ok.sum()>=15:
            exog_pred=sm.add_constant(pred_alineado[ok].to_numpy())
            row['BP_pred_p']=float(het_breuschpagan(e[ok].to_numpy(),exog_pred)[1])
            row['White_pred_p']=float(het_white(e[ok].to_numpy(),exog_pred)[1])
    except Exception as exc:
        row['error_diagnostico']=str(exc)
    return row

def comparar_modelos_adicionales(medidor,device='auto'):
    filas=[];predicciones={}
    for nombre,cfg_res in RESOLUCION_MODELO.items():
        freq,seasonal_periods=cfg_res['freq'],cfg_res['seasonal_periods']
        horizonte=HORIZONTE_DIAS*cfg_res['pasos_por_dia']
        started=time.time()
        try:
            serie=load_meter(medidor)['P'].resample(freq).mean().dropna()
            minimo=max(2*seasonal_periods,horizonte+seasonal_periods*2)
            if len(serie)<minimo:
                raise ValueError(f'Se requieren >= {minimo} periodos ({freq}) antes del test, hay {len(serie)}')
            train,test=serie.iloc[:-horizonte],serie.iloc[-horizonte:]
            kwargs={'seasonal_periods':seasonal_periods}
            if nombre=='LSTM': kwargs.update(device=device,epochs=80,hidden_size=32,seed=CFG['seed'])
            model=create_forecaster(nombre,**kwargs)
            if nombre=='SARIMAX':
                # PBI como unica exogena disponible con cobertura completa del horizonte
                # (feriados/temperatura son de calendario/clima, no de actividad economica;
                # PBI ya se usa igual en los 6 modelos ML operativos, seccion 12).
                exog_train=pd.DataFrame({'pbi_bcrp':pbi_en_fecha(train.index,PBI_MENSUAL)})
                exog_test=pd.DataFrame({'pbi_bcrp':pbi_en_fecha(test.index,PBI_MENSUAL)})
                model.fit(train,exog=exog_train)
                pred=pd.Series(model.predict(horizonte,exog=exog_test),index=test.index)
            else:
                model.fit(train)
                pred=pd.Series(model.predict(horizonte),index=test.index)
            if not np.isfinite(pred).all(): raise ValueError('Modelo genero valores no finitos')
            predicciones[nombre]=pred
            diag=_diagnosticos_residuales(test,pred,CFG['alfa_significancia'])
            filas.append(dict(modelo=nombre,frecuencia=freq,seasonal_periods=seasonal_periods,
                horizonte_pasos=horizonte,estado='OK',backend=getattr(model,'device_','CPU'),
                segundos=time.time()-started,**metrics(test,pred),**diag))
        except Exception as exc:
            filas.append(dict(modelo=nombre,frecuencia=freq,seasonal_periods=seasonal_periods,
                horizonte_pasos=horizonte,estado='ERROR',backend=None,
                segundos=time.time()-started,error=f'{type(exc).__name__}: {exc}'))
    return pd.DataFrame(filas),predicciones

MODELOS_ADICIONALES=[]
_con_datos=[m for m in TARGETS if any(WINNERS.get((m,v),{}).get('estado')==ESTADO_OK for v in ('P','Q','S_medida'))]
MEDIDORES_ADICIONALES=_con_datos[:CFG['max_medidores_adicionales']] if CFG.get('max_medidores_adicionales') else _con_datos
if MEDIDORES_ADICIONALES:
    for medidor in MEDIDORES_ADICIONALES:
        try:
            tabla,predicciones=comparar_modelos_adicionales(medidor,device='auto')
            tabla.insert(0,'medidor',medidor);MODELOS_ADICIONALES.append(tabla)
            for nombre,serie_pred in predicciones.items():
                serie_pred.rename('pred').to_csv(RUN/'pronosticos'/f'adicional_{TOKENS[medidor]}_{nombre}.csv')
        except Exception as exc:
            log('modelos_adicionales',medidor,'ERROR',exc)
if MODELOS_ADICIONALES:
    metricas_adicionales=pd.concat(MODELOS_ADICIONALES,ignore_index=True)
    metricas_adicionales.to_csv(RUN/'tablas/metricas_modelos_adicionales.csv',index=False)
    display(metricas_adicionales)
    print('Modelos adicionales (resolucion por modelo, con diagnosticos) calculados para',
          len(MODELOS_ADICIONALES),'medidor(es)')
